# Pension Model Results Analysis

This notebook analyses the run selected by `RUN_TAG` in `Results/Runs/`.

**Structure.** The cells above the crosswalk are infrastructure: run inventory,
output loading, and metric construction. They define everything the rest of the
notebook uses and are not analysis. Below them sits the mapping from the V6
draft's closing list to the Lenney et al. exhibits it refers to, which is the
specification for what this notebook is being rebuilt to produce. Everything
after that is the previous analysis, kept in an archive and sorted by how useful
it is against that specification.

A note on simulation structure: as of 2026-06-10 the Python asset simulation uses
**common market shocks**, so simulation column *n* is the same market history for
every plan and cross-plan aggregate distributions are meaningful. If the loaded
outputs predate this change, or are R outputs with independent per-plan draws,
aggregate bands understate risk; the load cell below checks the flag.

In [ ]:
from pathlib import Path
import sys

# results_analysis.py is co-located with this notebook
_here = Path.cwd().resolve()
if str(_here) not in sys.path:
    sys.path.insert(0, str(_here))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import results_analysis as ra

ROOT = ra.find_project_root()
RUN_TAG = "20260908_1"   # the baseline. Pinned deliberately: ra.latest_run_tag()
                         # would now return a counterfactual run (20260910_1..3),
                         # which would be analysed as the baseline with no error.
print(f"RUN_TAG = {RUN_TAG!r}")
RESULT_SOURCE = "auto"    # "auto" detects from the run folder; or set "rdata" (R) / "parquet" (Python)
GRAPH_YEARS = 15          # horizon for forecast-style figures; risk metrics use the full projection
RUN_DIR = ra.run_dir(ROOT, RUN_TAG)
if RESULT_SOURCE == "auto":
    RESULT_SOURCE = ra.detect_result_source(ROOT, RUN_TAG)
print(f"RESULT_SOURCE = {RESULT_SOURCE!r}")

plt.style.use("seaborn-v0_8-whitegrid")
print(ROOT)
print(RUN_DIR)

## Run Inventory

The first table counts deterministic A/L and asset-simulation statuses in the selected run manifest. The second table lists the plan-level manifest records, so missing or skipped outputs can be identified before interpreting the results.

In [ ]:
manifest = ra.load_manifest(ROOT, RUN_TAG)
status_counts = manifest.groupby(["detal_status", "asset_status"]).size().rename("count").reset_index()
display(status_counts)
display(manifest.sort_values(["asset_status", "detal_status", "plan"]))

## Select And Read Simulation Outputs

`RUN_TAG` selects the folder under `Results/Runs/`. `RESULT_SOURCE` sets the input format: `"auto"` (default) detects it from the run folder contents, or set `"rdata"` (R run outputs) / `"parquet"` (Python run outputs) explicitly. `SELECTED_PLANS = None` reads every plan in that run with an asset-simulation result; set it to a list such as `["CA10", "AZ06"]` for a faster subset.

For R result files (`RESULT_SOURCE = "rdata"`), the notebook creates clean `*_analysis.RData` companion files when needed. The cell also checks the common-market-shock flag saved with each plan's asset output: aggregate distribution bands later in the notebook are only meaningful when all plans share one shock matrix.

In [ ]:
SELECTED_PLANS = None
# SELECTED_PLANS = ["CA10", "AZ06", "PA93"]

if RESULT_SOURCE == "rdata":
    ra.prepare_analysis_exports(ROOT, RUN_TAG, plans=SELECTED_PLANS, overwrite=False, progress=True)

results = ra.load_run_results(ROOT, RUN_TAG, plans=SELECTED_PLANS, progress=True, source=RESULT_SOURCE)
print(f"Using {len(results)} plans with asset simulations in the selected run.")

# Common-shock flag check (see header note)
_common = {p: str(r.scalars.get("common_market_shocks", "")).strip().lower() == "true"
           for p, r in results.items()}
if all(_common.values()):
    seeds = {str(r.scalars.get("market_seed", "")) for r in results.values()}
    print(f"All plans share common market shocks (market_seed = {sorted(seeds)}). Aggregate fans are valid.")
else:
    missing = sorted(p for p, ok in _common.items() if not ok)
    print(f"[warn] {len(missing)} plan(s) lack the common-shock flag: {missing}")
    print("[warn] Aggregate distribution bands will UNDERSTATE risk (independent draws cancel across plans).")

ppd = ra.load_ppd(ROOT)
print(f"PPD reference data: {ppd.shape[0]} rows")

analysis_inventory = pd.DataFrame(
    {
        "plan": plan,
        "ppid": result.ppid,
        "plan_year": result.plan_year,
        "n_years": result.n_years,
        "n_simulations": result.n_simulations,
        "file": str(result.file_path),
    }
    for plan, result in results.items()
)
display(analysis_inventory)

## Metric Construction

This code-only section builds the plan-level variables used throughout: official funded ratios, unfunded liabilities relative to payroll, contribution rates, active-to-retired ratios, first asset-exhaustion years per simulation path, aggregate per-path balance sheets, and FRED fetch helpers. `N_PROJ` is the usable projection length (the final placeholder year with zero AAL is dropped).

In [ ]:
import json as _json
import os
import re
from urllib.parse import urlencode
from urllib.request import urlopen

# Population growth is a SIMULATION setting, not an analysis choice: the engine
# refills the workforce at this rate every year (engine/run_plan.py). It is read
# from the run rather than retyped here, because a hand-copied constant silently
# stops matching the moment the engine's value changes.
#
# Runs produced before 2026-07-31 do not carry it -- it was not saved with the
# results -- so those fall back to the value the engine used at the time, and the
# fallback announces itself rather than passing silently.
MODEL_POPULATION_GROWTH_FALLBACK = 0.01

_pop_values = {float(r.scalars["PopulationGrowth"])
               for r in results.values() if r.scalars.get("PopulationGrowth") is not None}
if not _pop_values:
    MODEL_POPULATION_GROWTH = MODEL_POPULATION_GROWTH_FALLBACK
    print(f"[note] this run predates 2026-07-31 and does not record PopulationGrowth; "
          f"using the engine value of {MODEL_POPULATION_GROWTH} for GDP projection")
elif len(_pop_values) > 1:
    raise ValueError(f"plans in this run disagree on PopulationGrowth: {sorted(_pop_values)}")
else:
    MODEL_POPULATION_GROWTH = _pop_values.pop()

# The disability payout rate is likewise a choice, and is what the paired
# sensitivity varies. Reported here so a run states which side of that pair it is.
_dis_values = {float(r.scalars["DisabilityPayoutRate"])
               for r in results.values() if r.scalars.get("DisabilityPayoutRate") is not None}
DISABILITY_PAYOUT_RATE = _dis_values.pop() if len(_dis_values) == 1 else None
if _dis_values:
    print(f"[note] plans in this run use different disability rates: {sorted(_dis_values)}")
BASE_YEAR = int(min(result.plan_year for result in results.values()))

# THE TWO HORIZONS NOW AGREE. They did not before 2026-08-04.
#
# Runs produced from 2026-08-04 onward carry `Nyear = 36`: the base year plus 35
# projected years, and every row of every matrix is filled. Both sides run
# BASE_YEAR .. BASE_YEAR + 35.
#
# Runs produced BEFORE that date carry `Nyear = 35` AND a liability loop that
# left its final row unwritten, so AAL, cash_inflows, cash_outflows and
# NormalCost held 34 real values while Assets held 35 -- the liability side
# stopped a year short of the asset side. The code below detects which kind of
# run it is loading rather than assuming, so older runs still analyse correctly.
#
#   N_PROJ      = rows with a usable liability  -> BASE_YEAR .. BASE_YEAR + N_PROJ - 1
#   MAX_OFFSET  = projected asset years          -> BASE_YEAR + 1 .. BASE_YEAR + MAX_OFFSET
#
# Horizons are named by fiscal year throughout, so any residual difference is
# visible in the column names instead of hidden behind a "years ahead" count.
_n_rows = min(r.n_years for r in results.values())
_trailing_zero = all(
    float(np.abs(r.matrix("AAL").to_numpy(dtype="float64")[-1]).max()) == 0.0
    for r in results.values())
N_PROJ = _n_rows - 1 if _trailing_zero else _n_rows
MAX_OFFSET = _n_rows - 1
if _trailing_zero:
    print(f"[note] this run predates 2026-08-04: its liability side stops at "
          f"{BASE_YEAR + N_PROJ - 1} while assets run to {BASE_YEAR + MAX_OFFSET}")

EXHAUST_HORIZONS = (10, 20, 30, MAX_OFFSET)
EXHAUST_YEARS = tuple(BASE_YEAR + h for h in EXHAUST_HORIZONS)
EXH_COLS = {h: f"prob_exhaust_by_{BASE_YEAR + h}" for h in EXHAUST_HORIZONS}
EXH_COL_MAX = EXH_COLS[MAX_OFFSET]
# Horizon used by the two cross-sectional scatters. Named rather than written
# inline so the choice is visible and consistent between them.
SCATTER_HORIZON = 20
LIABILITY_LAST_YEAR = BASE_YEAR + N_PROJ - 1
ASSET_LAST_YEAR = BASE_YEAR + MAX_OFFSET

TABLE1_METRICS = [
    ("Assets / liabilities", "assets_liabilities", "ActLiabilities_GASB"),
    ("Unfunded liabilities / payroll", "unfunded_liabilities_payroll", "payroll"),
    ("Total pension contributions / payroll", "total_pension_contributions_payroll", "payroll"),
    ("Active members / retired members", "active_retired_members", "retired_members"),
]
# "Projected active member growth" was removed from this table on 2026-07-31.
# It was not a plan characteristic: it is the model's own hardcoded workforce
# growth assumption, (1 + MODEL_POPULATION_GROWTH) ** 30 - 1, identical for
# every plan, so it reported a mean of 0.3478 and a standard deviation of
# exactly zero while sitting in a table of measured quantities. The assumption
# itself is stated in the markdown above the table.


def finite_numeric(series):
    values = pd.to_numeric(series, errors="coerce")
    return values.where(np.isfinite(values))


def num_col(frame, name):
    if name not in frame.columns:
        return pd.Series(np.nan, index=frame.index, dtype="float64")
    return finite_numeric(frame[name])


def safe_ratio(numerator, denominator):
    numerator = finite_numeric(numerator)
    denominator = finite_numeric(denominator)
    return (numerator / denominator).where(denominator > 0)


def first_available(frame, names):
    out = pd.Series(np.nan, index=frame.index, dtype="float64")
    for name in names:
        if name in frame.columns:
            out = out.fillna(pd.to_numeric(frame[name], errors="coerce"))
    return out


def add_core_metrics(frame):
    data = frame.copy()
    data["assets_liabilities"] = safe_ratio(num_col(data, "ActAssets_GASB"), num_col(data, "ActLiabilities_GASB"))
    data["unfunded_liabilities"] = num_col(data, "ActLiabilities_GASB") - num_col(data, "ActAssets_GASB")
    data["unfunded_liabilities_payroll"] = safe_ratio(data["unfunded_liabilities"], num_col(data, "payroll"))
    data["total_pension_contributions_payroll"] = safe_ratio(num_col(data, "contrib_tot"), num_col(data, "payroll"))
    data["retired_members"] = first_available(data, ["beneficiaries_tot", "beneficiaries_ServiceRetirees"])
    data["active_retired_members"] = safe_ratio(num_col(data, "actives_tot"), data["retired_members"])
    data["official_funded_ratio"] = data["assets_liabilities"]
    data["liability_billion"] = num_col(data, "ActLiabilities_GASB") / 1_000_000
    data["unfunded_liability_billion"] = data["unfunded_liabilities"] / 1_000_000
    data["contribution_rate"] = data["total_pension_contributions_payroll"]
    return data


def build_plan_metrics(results):
    rows = []
    for plan, result in sorted(results.items()):
        row = result.planinfo.iloc[0].to_dict() if result.planinfo is not None else {}
        row.update({
            "plan": plan,
            "ppid": result.ppid,
            "model_aal": result.scalars.get("Model_AAL"),
            "cafr_aal": result.scalars.get("CAFR_AAL"),
            "percent_difference": result.scalars.get("Percent_difference"),
            "n_simulations": result.n_simulations,
        })
        rows.append(row)
    data = add_core_metrics(pd.DataFrame(rows))
    data["model_aal_billion"] = pd.to_numeric(data["model_aal"], errors="coerce") / 1_000_000_000
    data["cafr_aal_billion"] = pd.to_numeric(data["cafr_aal"], errors="coerce") / 1_000_000_000
    data["percent_difference"] = pd.to_numeric(data["percent_difference"], errors="coerce")
    return data


def weighted_mean_sd(values, weights):
    values = finite_numeric(values).to_numpy(dtype="float64")
    weights = finite_numeric(weights).to_numpy(dtype="float64")
    mask = np.isfinite(values) & np.isfinite(weights) & (weights > 0)
    if not mask.any():
        return np.nan, np.nan, 0
    values = values[mask]
    weights = weights[mask]
    mean = np.average(values, weights=weights)
    sd = np.sqrt(np.average((values - mean) ** 2, weights=weights))
    return mean, sd, int(mask.sum())


def table1_summary(frame):
    rows = []
    for label, value_col, weight_col in TABLE1_METRICS:
        values = finite_numeric(frame[value_col])
        unweighted = values[np.isfinite(values)]
        weighted_mean, weighted_sd, weighted_obs = weighted_mean_sd(values, frame.get(weight_col))
        rows.append({
            "metric": label,
            "unweighted_mean": unweighted.mean(),
            "unweighted_sd": unweighted.std(ddof=1),
            "unweighted_obs": int(unweighted.shape[0]),
            "weighted_mean": weighted_mean,
            "weighted_sd": weighted_sd,
            "weighted_obs": weighted_obs,
        })
    return pd.DataFrame(rows)


def historical_official_funding(ppd, ppids):
    hist = ppd.loc[ppd["ppd_id"].isin(ppids)].copy()
    hist["fy"] = pd.to_numeric(hist["fy"], errors="coerce")
    hist["official_funded_ratio"] = safe_ratio(hist["ActAssets_GASB"], hist["ActLiabilities_GASB"])
    hist["liability_weight"] = num_col(hist, "ActLiabilities_GASB")
    rows = []
    for fy, group in hist.groupby("fy"):
        values = group["official_funded_ratio"]
        weights = group["liability_weight"]
        weighted_mean, _, weighted_obs = weighted_mean_sd(values, weights)
        rows.append({
            "fy": fy,
            "equal_weighted": values.mean(),
            "liability_weighted": weighted_mean,
            "observations": int(values.notna().sum()),
            "weighted_observations": weighted_obs,
        })
    return pd.DataFrame(rows).sort_values("fy")


def exhaustion_year(result):
    """Offset from the base year of the year in which assets reach zero.

    One value per simulated path; NaN where it never happens. Offsets run
    1..MAX_OFFSET, i.e. BASE_YEAR + 1 .. ASSET_LAST_YEAR.

    The value is unique per plan-path in the runs produced so far -- no path has
    two separate spells at zero -- so there is no "first" exhaustion to
    distinguish from a later one. Nothing in the model ENFORCES that, though:
    assets at zero return positive in any year where contributions exceed
    benefit outflows, and IL33 does exactly that in 2056-57. The argmax below is
    therefore still the right way to locate the year.
    """
    assets = result.matrix("Assets").to_numpy(dtype="float64")
    zero = assets[1:, :] <= 0
    any_hit = zero.any(axis=0)
    first_hit = zero.argmax(axis=0) + 1.0
    return np.where(any_hit, first_hit, np.nan)


def years_insolvent_mean(result, horizon=None):
    # Average number of projection years with zero assets, across paths
    assets = result.matrix("Assets").to_numpy(dtype="float64")
    n = assets.shape[0] if horizon is None else min(horizon + 1, assets.shape[0])
    return float(np.mean((assets[1:n, :] <= 0).sum(axis=0)))


def exhaustion_plan_summary(results, plan_metrics, horizons=EXHAUST_HORIZONS):
    metric_lookup = plan_metrics.set_index("plan")
    rows = []
    for plan, result in sorted(results.items()):
        offsets = exhaustion_year(result)
        row = {
            "plan": plan,
            "liability_billion": metric_lookup.loc[plan, "liability_billion"],
            "official_funded_ratio": metric_lookup.loc[plan, "official_funded_ratio"],
            "unfunded_liabilities_payroll": metric_lookup.loc[plan, "unfunded_liabilities_payroll"],
            "contribution_rate": metric_lookup.loc[plan, "contribution_rate"],
            "prob_no_exhaustion": np.nanmean(np.isnan(offsets) | (offsets > MAX_OFFSET)),
        }
        for horizon in horizons:
            col = f"prob_exhaust_by_{result.plan_year + horizon}"
            row[col] = np.nanmean(offsets <= horizon)
            row[f"expected_liability_exhaust_by_{result.plan_year + horizon}_billion"] = (
                row["liability_billion"] * row[col])
        exhausted = offsets[np.isfinite(offsets)]
        row["median_exhaustion_year_if_exhausted"] = np.nan if exhausted.size == 0 else int(result.plan_year + np.nanmedian(exhausted))
        row["mean_years_insolvent"] = years_insolvent_mean(result, horizon=N_PROJ)
        rows.append(row)
    return pd.DataFrame(rows)


def liability_weighted_exhaustion_bins(results, plan_metrics):
    edges = [(1, 10), (11, 20), (21, 30), (31, MAX_OFFSET)]
    bins = [(lo, hi, f"{BASE_YEAR + lo}-{BASE_YEAR + hi}") for lo, hi in edges]
    never_label = f"Never by {ASSET_LAST_YEAR}"
    metric_lookup = plan_metrics.set_index("plan")
    rows = []
    for plan, result in sorted(results.items()):
        offsets = exhaustion_year(result)
        liability = metric_lookup.loc[plan, "liability_billion"]
        for low, high, label in bins:
            prob = np.nanmean((offsets >= low) & (offsets <= high))
            rows.append({"bin": label, "plan": plan, "probability": prob, "liability_billion": liability, "expected_liability_billion": liability * prob})
        prob_never = np.nanmean(np.isnan(offsets) | (offsets > MAX_OFFSET))
        rows.append({"bin": never_label, "plan": plan, "probability": prob_never, "liability_billion": liability, "expected_liability_billion": liability * prob_never})
    data = pd.DataFrame(rows)
    total_liabilities = plan_metrics["liability_billion"].sum()
    summary = data.groupby("bin", as_index=False)["expected_liability_billion"].sum()
    summary["liability_share"] = summary["expected_liability_billion"] / total_liabilities
    order = [label for _, _, label in bins] + [never_label]
    summary["bin"] = pd.Categorical(summary["bin"], categories=order, ordered=True)
    # Bin widths are NOT equal -- the first three span ten years each and the last
    # spans four, because the projection ends at MAX_OFFSET rather than on a round
    # number. Comparing bin totals therefore compares different-length windows and
    # makes the final bin look small when its per-year rate is the highest of all.
    # The per-year column is the comparable one.
    widths = {label: hi - lo + 1 for lo, hi, label in bins}
    summary["years_in_bin"] = summary["bin"].astype(str).map(widths)
    summary["expected_liability_per_year_billion"] = (
        summary["expected_liability_billion"] / summary["years_in_bin"])
    return summary.sort_values("bin")


def liability_weighted_exhaustion_by_year(results, plan_metrics):
    """Expected liability of plans exhausting in EACH projected year.

    No binning: one row per fiscal year, so no window-width choice can distort
    the shape. Plans are weighted by base-year GASB liability, and the value is
    that liability times the probability that the plan exhausts in that year.
    The mass that never exhausts is returned separately, since it belongs to no
    year and cannot be drawn on the same axis honestly.
    """
    lookup = plan_metrics.set_index("plan")["liability_billion"]
    years = np.arange(1, MAX_OFFSET + 1)
    expected = np.zeros(len(years), dtype="float64")
    never = 0.0
    for plan, result in sorted(results.items()):
        offsets = exhaustion_year(result)
        liability = lookup.get(plan, np.nan)
        if not np.isfinite(liability):
            continue
        for j, horizon in enumerate(years):
            expected[j] += liability * np.nanmean(offsets == horizon)
        never += liability * np.nanmean(np.isnan(offsets) | (offsets > MAX_OFFSET))
    total = float(lookup.sum())
    frame = pd.DataFrame({
        "year": BASE_YEAR + years,
        "expected_liability_billion": expected,
        "liability_share": expected / total,
    })
    return frame, never, total


def liability_weighted_exhausted_by_year(results, plan_metrics):
    """Expected liability that HAS exhausted assets by each projected year.

    The cumulative counterpart of `liability_weighted_exhaustion_by_year`: that
    one counts a plan once, in the year it first runs out; this one counts it in
    every year from then on, so it is a stock rather than a flow.

    The two are the same information arranged differently, and this one is also
    a rescaling of the liability-weighted curve in the exhaustion-CDF figure
    below -- that curve as a probability, this one multiplied by total
    liabilities to give dollars.

    Verified on this run: no path ever returns to positive assets after reaching
    zero (0 recoveries out of 132,127 exhausting paths), so "has exhausted by
    year t" and "is exhausted in year t" are the same series. If a future engine
    change ever lets a plan recover, they separate and this function measures
    the first of the two.
    """
    lookup = plan_metrics.set_index("plan")["liability_billion"]
    years = np.arange(1, MAX_OFFSET + 1)
    expected = np.zeros(len(years), dtype="float64")
    for plan, result in sorted(results.items()):
        offsets = exhaustion_year(result)
        liability = lookup.get(plan, np.nan)
        if not np.isfinite(liability):
            continue
        for j, horizon in enumerate(years):
            expected[j] += liability * np.nanmean(offsets <= horizon)
    total = float(lookup.sum())
    never = total - float(expected[-1])
    frame = pd.DataFrame({
        "year": BASE_YEAR + years,
        "expected_liability_billion": expected,
        "liability_share": expected / total,
    })
    return frame, never, total


def exhaustion_cdf(results, plan_metrics, max_year=35):
    # P(exhaustion by year y) per plan, plus the liability-weighted average curve
    lookup = plan_metrics.set_index("plan")["liability_billion"]
    horizons = np.arange(1, max_year + 1)
    per_plan = {}
    for plan, result in sorted(results.items()):
        offsets = exhaustion_year(result)
        per_plan[plan] = np.array([np.nanmean(offsets <= h) for h in horizons])
    cdf = pd.DataFrame(per_plan, index=pd.Index(horizons, name="years_ahead"))
    weights = lookup.reindex(cdf.columns).to_numpy(dtype="float64")
    weights = np.where(np.isfinite(weights), weights, 0.0)
    weighted = cdf.to_numpy() @ weights / weights.sum()
    return cdf, pd.Series(weighted, index=cdf.index, name="liability_weighted")


def aggregate_matrix(results, matrix_name):
    """Sum one matrix across plans, path by path.

    Uses a plain sum, not nansum: a NaN in any plan must propagate into the
    aggregate rather than being silently treated as a zero contribution. The
    check below turns that into an explicit error naming the plans involved.
    """
    arrays = [result.matrix(matrix_name).to_numpy(dtype="float64") for result in results.values()]
    min_years = min(arr.shape[0] for arr in arrays)
    min_sims = min(arr.shape[1] for arr in arrays)
    clipped = [arr[:min_years, :min_sims] for arr in arrays]
    bad = [plan for plan, arr in zip(results, clipped) if np.isnan(arr).any()]
    if bad:
        raise ValueError(
            f"{matrix_name} contains NaN for {len(bad)} plan(s): {bad}. "
            "Aggregating would hide it; investigate the run before continuing.")
    return np.sum(np.stack(clipped, axis=0), axis=0)


def aggregate_paths(results):
    # Per-path aggregate balance sheet. Distributionally meaningful only with
    # common market shocks across plans (checked at load time above).
    assets = aggregate_matrix(results, "Assets")
    aal = aggregate_matrix(results, "AAL")
    first = next(iter(results.values()))
    years = np.asarray(first.years(assets.shape[0]))
    valid = np.any(np.isfinite(aal) & (aal > 0), axis=1)
    return years[valid], assets[valid, :], aal[valid, :]


def fan_table(years, matrix, quantiles=(0.05, 0.25, 0.50, 0.75, 0.95)):
    out = {"year": [int(y) for y in years], "mean": np.nanmean(matrix, axis=1)}
    for q in quantiles:
        out[f"q{int(round(q * 100)):02d}"] = np.nanquantile(matrix, q, axis=1)
    return pd.DataFrame(out)


def plot_fan(ax, fan, label, color="tab:blue"):
    x = fan["year"].to_numpy()
    ax.fill_between(x, fan["q05"], fan["q95"], alpha=0.12, color=color, label="5-95 pct")
    ax.fill_between(x, fan["q25"], fan["q75"], alpha=0.22, color=color, label="25-75 pct")
    ax.plot(x, fan["mean"], color=color, label=f"{label} mean")
    ax.plot(x, fan["q50"], color=color, linestyle="--", linewidth=1, label=f"{label} median")


# --- FRED helpers (used by the GDP and AAA sections; both skip gracefully) ---
# Read from the environment first, then from the untracked `.env` file at the
# project root. `.env` is gitignored, so the key never reaches GitHub; put it
# there once and every session picks it up with nothing to set by hand.
# See `.env.example` for the format.
FRED_API_KEY = ra.local_setting("FRED_API_KEY")
FRED_API_BASE_URL = "https://api.stlouisfed.org/fred/series/observations"


def fetch_fred_series(series_id, api_key=None, observation_start="1900-01-01"):
    api_key = api_key or FRED_API_KEY
    if not api_key:
        raise ValueError("FRED_API_KEY is not set (notebook variable or environment variable).")
    params = {"series_id": series_id, "api_key": api_key, "file_type": "json",
              "observation_start": observation_start}
    with urlopen(f"{FRED_API_BASE_URL}?{urlencode(params)}", timeout=30) as response:
        payload = _json.load(response)
    if "observations" not in payload:
        raise ValueError(f"FRED API did not return observations for {series_id}: {payload.get('error_message', payload)}")
    data = pd.DataFrame(payload["observations"])
    if data.empty:
        return pd.DataFrame(columns=["date", series_id])
    out = data[["date", "value"]].rename(columns={"value": series_id})
    out["date"] = pd.to_datetime(out["date"], errors="coerce")
    out[series_id] = pd.to_numeric(out[series_id].replace(".", np.nan), errors="coerce")
    return out.dropna(subset=["date", series_id]).sort_values("date")


def try_fetch_fred(series_id):
    """Fetch a FRED series, or return None after saying loudly that it failed.

    Sections built on FRED produce nothing at all without a key, so the failure
    is announced rather than mentioned: on a machine without FRED_API_KEY set,
    a quiet one-line note is easy to scroll past and leaves the notebook
    looking complete when two whole sections are missing.
    """
    try:
        return fetch_fred_series(series_id)
    except Exception as exc:
        bar = "!" * 78
        print(bar)
        print(f"SECTION SKIPPED -- FRED series {series_id} could not be fetched.")
        print(f"  reason : {exc}")
        print("  effect : every figure and table in this section is MISSING, not empty.")
        print("  fix    : put FRED_API_KEY in the .env file at the project root")
        print("           (copy .env.example to .env and fill it in), then rerun this cell.")
        print(bar)
        return None


plan_metrics = build_plan_metrics(results)
exhaustion = exhaustion_plan_summary(results, plan_metrics)
agg_years, agg_assets, agg_aal = aggregate_paths(results)
print(f"Metric frame: {len(plan_metrics)} modeled plans; usable projection years: {N_PROJ}")
print(f"Model settings recorded with this run: population growth "
      f"{MODEL_POPULATION_GROWTH:.3f}/year, disability payout rate "
      f"{DISABILITY_PAYOUT_RATE if DISABILITY_PAYOUT_RATE is not None else 'not recorded'}")

# The exhibits to reproduce, from Lenney et al.

The V6 list's numbers are the published article's numbers, one for one: its "Figure 6"
is Figure 6 of `Papers/Brookings papers/BPEA_SP21_FINAL_Lenney-et-al.pdf`. Read the
list only against that version; in the 25 March 2021 conference draft the same charts
are numbered two higher. Where the V6 list says "Figure 2 notes above" it means
**Table 2** — there is no Figure 2 note.

| V6 item | Lenney exhibit | What it shows | Modification needed |
|---|---|---|---|
| **Table 1** | Table 1, Estimation Sample of State and Local Pension Plans | Assets/liabilities, unfunded liabilities/payroll, total pension contributions/payroll, active members/retired members, projected percent active member growth, and number of observations. Estimation sample against the Public Plans Database and a national sample, reported both equally weighted and weighted. | Same set of statistics. They do weighted and equally weighted; we could do the same or not. |
| **Table 2** | Table 2, Percentage Point Increase in Contribution Rate Required (percent of payroll) | The percentage point increase in the contribution rate required, for different stabilization goals, as a function of the assumed real rate of return and of when the change starts. | Our metric in the same spirit: the percentage point increase required to bring the probability of experiencing a shortfall within the next 20 years below 0.5%, 1% and 3%. Shortfall over a period is a **cumulative** notion — counting the fraction of sample paths that ended in insolvency over each horizon. This is what makes sense to show plan by plan. Not sure it is a good number to report unconditionally across plans; maybe report it in aggregate for plans underfunded by some minimum percentage, e.g. 20%. See also the Figure 6 notes. |
| **Figure 1** | Figure 1, Funding Ratios under the AAA Corporate Bond Interest Rate | Average S&L funding ratios using AAA corporate bond rates, 2002 to 2018. Their numbers come from the Financial Accounts of the United States, citing Hoops, Smith and Stefanescu (2016). | Show both official and AAA-discounted funding ratios. Can May help get this from historical data, BC, Rauh, or elsewhere? Possible addition: compare the number of surprise crises under the different measures, to support the idea that the AAA ratio is somehow more telling — though if one is just a stable fraction of the other it would not make much difference. |
| **Figures 2, 3, 4** | Figure 2, Ratio of Beneficiaries to Active Workers; Figure 3, Ratio of Benefit Payments to GDP; Figure 4, the same under different scenarios | Macro and demographic paths: the beneficiary-to-worker ratio over time, aggregate benefit payments as a share of GDP, and that same series under COLA and new-hire-reform scenarios. | **None.** These are macro and demographics; they do not need revisiting and are reproduced as they are. |
| **Figure 5** | Figure 5, US Ratio of Assets to GDP | Pension assets to GDP under each assumed return, with contributions held at their current share of payroll. | **Not needed.** Not relevant because we are not assessing the aggregate burden, and because of the problems with the asset return assumptions in the original. |
| **Figure 6** | Figure 6, Percent of Total Liabilities in Plans That Exhaust Their Assets over Various Time Horizons | Percent of total liabilities in plans that exhaust their assets over various time horizons, as a function of the constant assumed rate of return. | Show percentage of total liabilities **and** value of total liabilities in plans that exhaust in different years, as a good way to illustrate aggregate magnitudes. Unlike the Table 2 approach it does not require looking only at plans already below some funding ratio. Overall paper emphasis will be more on the Table 2 cross-section. |
| **Figures 7, 8** | Figure 7, US Implicit Pension Debt under Pension Debt Stabilization; Figure 8, US Pension Assets under the same, both started at different time horizons | Liabilities to GDP, and the asset companion, under different start dates for stabilizing that ratio with a permanent increase in contributions that accomplishes stabilization. | Addressed by the Table 2 approach: how much contribution rates would have to rise to keep the likelihood of exhaustion over some horizon below some threshold. Ask how much higher future contributions would have to be to stabilize under (1) a variety of investment strategies, and (2) a variety of waiting periods before contributions start increasing — immediate, 5, 10, 15 and 20 years. |
| **Figure 9** | Figure 9, Distribution of Liabilities by Percentage Point Change in Contribution | Distribution of contribution rate increases, as a share of actuarial liabilities, required to achieve different measures of stability. | Similarly show the distribution of contribution rate increases required to achieve a threshold probability for solvency over our preferred horizon. |
| **Figure 10** | Figure 10, Required Contribution to Stabilize | Scatterplot of the needed funding increase to achieve the stability goal against the funded ratio, with a regression line. Their point is that it is not the worst-funded plans that must contribute most. | The more interesting one. Do the analogous plots for whatever we choose to define as stability targets, and to see how this varies with different investment strategies. |
| **Figures 11, 12** | Figure 11, Effects of Changes in Benefits and Contributions on Required Contribution; Figure 12, Required Contribution Lower in Plans That Have Made Large Changes | Effects of pension plan reforms on what is needed for stabilization: had recent reforms not happened, how much more would contributions have to be raised to stabilize. Reported by plan, split between benefit reforms undertaken and changes in contribution rates since 2007. Figure 12 gets at the same sort of thing. | Interesting. We will have to discuss how we want to measure policy changes — for instance, we could look at what would happen without tier changes after some date. Open question: do we have their data on changes in contribution rates over time? |

## Exhibit inputs

Exhibit inputs: the contribution grid, reduced to what the exhibits need.

Reading every scenario's Assets matrix takes roughly 25 minutes, so the reduced
form is cached to `output/exhibit_cache.pkl` and reused. Delete that file to
rebuild after a new run. Each scenario collapses to one number per plan per
year, the probability that the plan has hit zero assets at any point up to that
year, which is all any exhibit below needs.

In [ ]:
import json, pickle, re
from pathlib import Path

BASE_YEAR = int(next(iter(results.values())).plan_year)
CACHE = Path.cwd() / "output" / "exhibit_cache.pkl"


def discover_contribution_grid(root, baseline_tag):
    """(increase pp, start year) -> run tag, for scenarios built on this baseline."""
    runs = Path(root) / "Results" / "Runs"
    grid = {(0.0, 0): baseline_tag}
    for d in sorted(runs.iterdir()):
        if not d.is_dir() or d.name.startswith("_") or d.name == baseline_tag:
            continue
        f = next(iter(d.glob("*/*_parquet/scalars.parquet")), None)
        if f is None:
            continue
        scal = dict(zip(*pd.read_parquet(f)[["name", "value"]].to_numpy().T))
        raw = scal.get("scenario_json")
        if not raw:
            continue
        sc = json.loads(raw)
        add = float(sc.get("contrib_add") or 0.0)
        if add <= 0 or sc.get("detal_run_tag") != baseline_tag:
            continue
        grid[(add, int(sc.get("policy_start") or 0))] = d.name
    return dict(sorted(grid.items()))


def _cumulative_exhaustion(root, tag, plans):
    """P(plan has hit zero assets by year t), shape (n_plans, n_years)."""
    runs = Path(root) / "Results" / "Runs"
    out = None
    for i, plan in enumerate(plans):
        bundle = next(iter((runs / tag / plan).glob("*_parquet")))
        a = pd.read_parquet(bundle / "Assets.parquet").to_numpy(dtype="float64")
        if out is None:
            out = np.zeros((len(plans), a.shape[0]))
        out[i] = np.maximum.accumulate(a <= 0, axis=0).mean(axis=1)
    return out


GRID_PLANS = sorted(results)

if CACHE.exists():
    _c = pickle.load(open(CACHE, "rb"))
    CONTRIB_GRID, CUM = _c["grid"], _c["cum"]
    print(f"exhibit inputs loaded from {CACHE.name}")
else:
    CONTRIB_GRID = discover_contribution_grid(ROOT, RUN_TAG)
    CUM = {k: _cumulative_exhaustion(ROOT, t, GRID_PLANS) for k, t in CONTRIB_GRID.items()}
    CACHE.parent.mkdir(parents=True, exist_ok=True)
    pickle.dump({"grid": CONTRIB_GRID, "cum": CUM}, open(CACHE, "wb"))
    print(f"exhibit inputs built and cached to {CACHE.name}")

GRID_DELTAS = np.array(sorted({d for d, _ in CONTRIB_GRID}), dtype=float)
GRID_STARTS = np.array(sorted({s for _, s in CONTRIB_GRID}), dtype=int)
N_YEARS = CUM[(0.0, 0)].shape[1]

# Year-0 accrued liability per plan: the weight for every liability-weighted exhibit.
LIAB = plan_metrics.set_index("plan").reindex(GRID_PLANS)["liability_billion"].to_numpy(float)
LIAB = np.where(np.isfinite(LIAB), LIAB, 0.0)

print(f"  {len(GRID_PLANS)} plans, {N_YEARS} years ({BASE_YEAR}-{BASE_YEAR + N_YEARS - 1})")
print(f"  increases (pp of payroll): {list(GRID_DELTAS)}")
print(f"  start years:               {list(GRID_STARTS)}")
print(f"  total accrued liability:   ${LIAB.sum():,.0f}bn")


def risk_at(key, offset):
    """P(exhausted by `offset` years after the base year), one value per plan."""
    return CUM[key][:, offset]


def required_increase(deltas, risks, target):
    """Smallest increase reaching `target`, interpolated between grid points.

    Returns inf when the target is not reached anywhere in the grid. That is a
    result, not a missing value: the plan cannot get there on a contribution
    increase inside the range we regard as a policy.
    """
    r = np.asarray(risks, dtype=float)
    if r[0] <= target:
        return 0.0
    hit = np.nonzero(r <= target)[0]
    if hit.size == 0:
        return np.inf
    j = hit[0]
    r0, r1, d0, d1 = r[j - 1], r[j], deltas[j - 1], deltas[j]
    return float(d1) if r0 == r1 else float(d0 + (r0 - target) * (d1 - d0) / (r0 - r1))


def inversion(offset, targets, start=0):
    """Required increase per plan per target, at one policy start year."""
    keys = [(d, start if d > 0 else 0) for d in GRID_DELTAS]
    R = np.column_stack([risk_at(k, offset) for k in keys])
    df = pd.DataFrame({f"{t:.1%}": [required_increase(GRID_DELTAS, R[i], t)
                                    for i in range(len(GRID_PLANS))] for t in targets},
                      index=GRID_PLANS)
    df.index.name = "plan"
    return df, R

# Counterfactual reductions, added 2026-09-10. Same shape as the baseline grid:
#   NR_*        the contribution grid on the no-reform liabilities (Lenney 11/12)
#   CF_SERIES   per-plan paths for the COLA counterfactual runs (Lenney 4)
# Absent from an older cache, in which case the two exhibits below will say so
# rather than fail.
NR_GRID = _c.get("nr_grid") if CACHE.exists() else None
NR_CUM = _c.get("nr_cum") if CACHE.exists() else None
CF_SERIES = _c.get("cf_series") if CACHE.exists() else None
if NR_CUM is None or CF_SERIES is None:
    print("  NOTE: counterfactual reductions absent from the cache; "
          "delete output/exhibit_cache.pkl and rerun to build them")
else:
    print(f"  counterfactual grid: {len(NR_GRID)} points on {_c['nr_base']}; "
          f"COLA runs: {', '.join(_c['cola_runs'].values())}")

# Workforce growth as the run itself recorded it, rather than a hand-typed constant.
POP_GROWTH = float(next(iter(results.values())).scalars.get("PopulationGrowth", 0.01))
print(f"  population growth recorded in the run: {POP_GROWTH:.3f}")

# The exhibits

Each section below is one item from the list above, in the V6 list's own order.
Section headings name the Lenney exhibit they replace, so a reader can hold the two
side by side.

**The horizon.** Two are used, and they are not interchangeable. Twenty years, to
2042, is the horizon the V6 note proposes for the shortfall-probability target. The
full projection runs to 2057. Any exhibit involving a waiting period has to use the
full horizon, because a contribution increase starting in year 20 cannot affect
whether a plan survives the first twenty years.

**Not built here.** Lenney's Figure 4 needs COLA and new-hire-reform scenario runs,
and Figures 11 and 12 need the no-reform tier workbook, which does not exist yet.
Both are noted in the final section rather than silently omitted.

## Table 2 — Percentage point increase in contribution rate required

Theirs answers "what permanent increase stabilises the debt", as a function of an
assumed deterministic return. Ours answers the V6 note's version: what permanent
increase holds the probability of a shortfall within the next 20 years below 0.5%,
1% and 3%. Shortfall is cumulative, the fraction of simulated paths that have hit
zero assets at any point by that year.

In [ ]:
HORIZON_SHORT = 20              # years past the base year; the V6 note's horizon
TARGETS = (0.005, 0.01, 0.03)

req, R0 = inversion(HORIZON_SHORT, TARGETS, start=0)

TARGET_COL = f"target: P(shortfall by {BASE_YEAR + HORIZON_SHORT})"

rows = []
for col in req.columns:
    v = req[col]
    reached = v[np.isfinite(v)]
    w = LIAB[[GRID_PLANS.index(p) for p in reached.index]]
    missed = v[~np.isfinite(v)].index
    rows.append({
        TARGET_COL: f"below {col}",
        "plans reaching it": f"{len(reached)} of {len(v)}",
        "median increase": f"{reached.median():.2f}pp",
        "liability-weighted mean": f"{np.average(reached, weights=w):.2f}pp",
        "largest needed": f"{reached.max():.2f}pp",
        "liability in plans that cannot":
            f"{LIAB[[GRID_PLANS.index(p) for p in missed]].sum() / LIAB.sum():.1%}",
    })
table2 = pd.DataFrame(rows).set_index(TARGET_COL)
display(table2)

unreachable = req[~np.isfinite(req[f"{TARGETS[1]:.1%}"])].index.tolist()
print(f"Cannot reach {TARGETS[1]:.1%} on any increase up to "
      f"+{GRID_DELTAS.max():g}pp of payroll: {', '.join(unreachable)}")
print("For those plans the answer is that a contribution increase inside the "
      "plausible range does not buy the target, not that the number is missing.")

## Figure 6 — Share of total liabilities in plans that exhaust their assets

Theirs buckets plans by exhaustion horizon, one bar series per assumed deterministic
return, so each bar answers "if returns are exactly 2.5% forever". Ours is indexed by
probability instead: every plan exhausts on some paths and not others, so the
quantity plotted is the expected share of total liabilities sitting in plans that
have run out by each year. The V6 note asks for the dollar value alongside the share,
so both panels are drawn.

In [ ]:
offsets = np.arange(1, N_YEARS)
share = np.array([float((LIAB * CUM[(0.0, 0)][:, t]).sum() / LIAB.sum()) for t in offsets])
value = np.array([float((LIAB * CUM[(0.0, 0)][:, t]).sum()) for t in offsets])
years = BASE_YEAR + offsets

fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))
axes[0].fill_between(years, share * 100, color="#4C72B0", alpha=0.85)
axes[0].set_title("Share of total liabilities in plans that have exhausted")
axes[0].set_ylabel("percent of total accrued liability")
axes[1].fill_between(years, value, color="#55A868", alpha=0.85)
axes[1].set_title("Value of those liabilities")
axes[1].set_ylabel("$bn of accrued liability")
for ax in axes:
    ax.set_xlabel("year")
    ax.axvline(BASE_YEAR + HORIZON_SHORT, color="0.35", lw=1, ls="--")
    ax.annotate(f"{BASE_YEAR + HORIZON_SHORT}", (BASE_YEAR + HORIZON_SHORT, ax.get_ylim()[1]),
                xytext=(3, -12), textcoords="offset points", color="0.35", fontsize=9)
plt.tight_layout(); plt.show()

print(f"{'year':>6}{'horizon':>9}{'% of liability':>16}{'$bn':>10}")
for t in (10, 15, 20, 25, 30, N_YEARS - 1):
    print(f"{BASE_YEAR + t:>6}{t:>8}y{share[t - 1]:>15.1%}{value[t - 1]:>10,.0f}")

## Figures 7 and 8 — Waiting before the increase starts

Theirs plots debt to GDP and assets to GDP under a permanent increase begun today or
in 10, 20 or 30 years. Ours asks the V6 note's question instead: how much higher the
contribution has to be to hold exhaustion risk below a threshold, and how that cost
rises with waiting.

Measured over the **full projection to 2057**, not the 20-year horizon used above. A
contribution increase that starts in year 20 cannot change whether a plan survives
its first twenty years, so the short horizon makes the delayed cases meaningless.

In [ ]:
HORIZON_LONG = N_YEARS - 1
TARGET_ONE = 0.01

rows = []
for start in GRID_STARTS:
    d, _ = inversion(HORIZON_LONG, (TARGET_ONE,), start=int(start))
    v = d.iloc[:, 0]
    ok = v[np.isfinite(v)]
    w = LIAB[[GRID_PLANS.index(p) for p in ok.index]]
    rows.append({"start year": int(start),
                 "plans reaching target": len(ok),
                 "median increase (pp)": round(float(ok.median()), 2) if len(ok) else np.nan,
                 "liability-weighted (pp)": round(float(np.average(ok, weights=w)), 2) if len(ok) else np.nan,
                 "liability that cannot (%)": round(100 * LIAB[[GRID_PLANS.index(p) for p in v[~np.isfinite(v)].index]].sum() / LIAB.sum(), 1)})
waiting = pd.DataFrame(rows).set_index("start year")
display(waiting)

fig, ax = plt.subplots(figsize=(7.5, 4.4))
ax.plot(waiting.index, waiting["median increase (pp)"], "o-", label="median plan")
ax.plot(waiting.index, waiting["liability-weighted (pp)"], "s--", label="liability-weighted")
ax.set_xlabel("years waited before contributions rise")
ax.set_ylabel(f"increase needed for P(exhaustion by {BASE_YEAR + HORIZON_LONG}) < {TARGET_ONE:.0%}")
ax.set_title("Cost of waiting")
ax.legend(); plt.tight_layout(); plt.show()

## Figure 9 — Distribution of liabilities by required contribution increase

Theirs is three panels of liability-weighted histograms, one per stabilisation goal.
Ours keeps the liability weighting, which is what makes the figure about exposure
rather than plan counts, and replaces the goal with the shortfall-probability target.
Plans that cannot reach the target at any increase in the grid are drawn as a
separate bar rather than dropped, so the sample is not silently truncated.

In [ ]:
edges = [0, 2.5, 5, 10, 17.5]
labels = ["0", "0-2.5", "2.5-5", "5-10", "10-17.5", "cannot reach"]

fig, axes = plt.subplots(1, len(TARGETS), figsize=(14, 4.2), sharey=True)
for ax, t in zip(np.atleast_1d(axes), TARGETS):
    v = req[f"{t:.1%}"]
    w = LIAB
    buckets = np.zeros(len(labels))
    for i, val in enumerate(v.to_numpy(dtype=float)):
        if not np.isfinite(val):
            buckets[-1] += w[i]
        elif val == 0:
            buckets[0] += w[i]
        else:
            buckets[int(np.searchsorted(edges, val, side="left"))] += w[i]
    ax.bar(labels, 100 * buckets / w.sum(),
           color=["#55A868"] + ["#4C72B0"] * 4 + ["#C44E52"])
    ax.set_title(f"target P < {t:.1%}")
    ax.tick_params(axis="x", rotation=45)
axes[0].set_ylabel("percent of total accrued liability")
fig.suptitle(f"Liabilities by the increase needed, within {HORIZON_SHORT} years", y=1.02)
plt.tight_layout(); plt.show()

## Figure 10 — Required contribution against funded ratio

Theirs scatters the required contribution change against the funded ratio, one point
per plan, making the point that it is not the worst-funded plans that must contribute
most. Ours puts our own target on the vertical axis. Plans that cannot reach the
target at any increase in the grid are drawn one step above the grid ceiling in a
different colour and marked, rather than dropped.

In [ ]:
fr = plan_metrics.set_index("plan").reindex(GRID_PLANS)
x = fr["official_funded_ratio"].to_numpy(dtype=float)   # the plans own reported ratio
y = req[f"{TARGET_ONE:.1%}"].to_numpy(dtype=float)
cap = GRID_DELTAS.max() * 1.15
unreach = ~np.isfinite(y)

fig, ax = plt.subplots(figsize=(8.4, 5.4))
ax.scatter(x[~unreach], y[~unreach], s=np.sqrt(LIAB[~unreach]) * 4,
           alpha=0.75, color="#4C72B0", label="reaches the target")
ax.scatter(x[unreach], np.full(unreach.sum(), cap), s=np.sqrt(LIAB[unreach]) * 4,
           alpha=0.85, color="#C44E52", marker="^", label=f"cannot within +{GRID_DELTAS.max():g}pp")
for i, p in enumerate(GRID_PLANS):
    if unreach[i] or y[i] > 8 or LIAB[i] > 150:
        ax.annotate(p, (x[i], cap if unreach[i] else y[i]), fontsize=7,
                    xytext=(4, 3), textcoords="offset points")
good = np.isfinite(x) & np.isfinite(y)
if good.sum() > 2:
    b, a = np.polyfit(x[good], y[good], 1)
    xs = np.linspace(np.nanmin(x), np.nanmax(x), 50)
    ax.plot(xs, a + b * xs, color="0.35", lw=1.2, ls="--",
            label=f"fit, slope {b:.1f}pp per unit funded ratio")
ax.axhline(cap, color="0.8", lw=0.8)
ax.set_xlabel("funded ratio")
ax.set_ylabel(f"increase needed for P(shortfall by {BASE_YEAR + HORIZON_SHORT}) < {TARGET_ONE:.0%}")
ax.set_title("Required contribution increase against funded ratio")
ax.legend(fontsize=8); plt.tight_layout(); plt.show()
print("Marker area is proportional to accrued liability.")

## Figures 2 and 3 — Beneficiaries per active worker, and benefit payments

The V6 list marks Lenney's Figures 2, 3 and 4 as needing no modification: macro and
demographics, reproduced as they are. Figures 2 and 3 are drawn here from the
population and benefit-payment series the engine began saving on 2026-09-08. Figure 4
needs COLA and new-hire-reform runs and is not built.

Two differences from theirs are structural rather than presentational. Theirs runs to
2117 and covers the whole US state and local system; ours runs to 2057 over these 40
plans. And the active headcount here is the projection's own, which is not a constant
1% growth path in aggregate, because only the newest tier hires while older tiers
wind down.

In [ ]:
act = np.vstack([results[p].matrix("active_members").to_numpy(dtype=float)[:, 0] for p in GRID_PLANS])
ben = np.vstack([results[p].matrix("beneficiaries").to_numpy(dtype=float)[:, 0] for p in GRID_PLANS])
pay = np.vstack([results[p].matrix("benefit_payments").to_numpy(dtype=float)[:, 0] for p in GRID_PLANS])
yrs = BASE_YEAR + np.arange(act.shape[1])

fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))
axes[0].plot(yrs, ben.sum(axis=0) / act.sum(axis=0), color="#4C72B0", lw=2)
axes[0].set_title("Beneficiaries per active worker, all 40 plans")
axes[0].set_ylabel("beneficiaries / actives")
axes[1].plot(yrs, pay.sum(axis=0) / 1e9, color="#55A868", lw=2)
axes[1].set_title("Benefit payments")
axes[1].set_ylabel("$bn per year")
for ax in axes:
    ax.set_xlabel("year")
plt.tight_layout(); plt.show()

print(f"{'year':>6}{'actives':>12}{'beneficiaries':>15}{'ratio':>8}{'benefits $bn':>14}")
for t in (0, 10, 20, act.shape[1] - 1):
    print(f"{yrs[t]:>6}{act[:, t].sum():>12,.0f}{ben[:, t].sum():>15,.0f}"
          f"{ben[:, t].sum() / act[:, t].sum():>8.2f}{pay[:, t].sum() / 1e9:>14.1f}")

## Table 1 — Estimation sample of state and local pension plans

Theirs reports assets/liabilities, unfunded liabilities/payroll, contributions/payroll,
actives/retirees, projected active member growth and the observation count, for the 40
plans against the PPD and a national sample, both equally weighted and weighted. The V6
note leaves open whether we report both weightings.

Their own values are in `Data/Sources/brookings_bpea2021_replication/Main paper/Data &
Programs/Table 1/table1_data.xlsx`, so ours can be shown beside theirs rather than
described.

Note on the growth row: theirs is a plan-level input, ours is the engine's fixed 1% a
year applied identically to every plan, so it carries no cross-plan variation and is
reported as an assumption rather than a measurement.

In [ ]:
baseline_summary = table1_summary(plan_metrics)
display(baseline_summary.round(3))

core_plan_table = plan_metrics[[
    "plan", "PlanName", "official_funded_ratio", "unfunded_liabilities_payroll",
    "total_pension_contributions_payroll", "active_retired_members",
    "liability_billion", "unfunded_liability_billion", "n_simulations"
]].sort_values("official_funded_ratio")
display(core_plan_table.round(3))

## Figure 1 — Funding ratios under the AAA corporate bond rate

Theirs is one historical line, 2002 to 2018, from the Financial Accounts of the United
States. The V6 note asks for the official and the AAA-discounted ratio together; the
official counterpart is their own online appendix Figure A3, so it needs no new
sourcing, and their plotted series is in the replication package.

Two panels below: the historical official ratios from the PPD, then the AAA-discounted
present value of our projected cash flows. The second is a different object from theirs
— a revaluation of our own model output rather than a history — so they are drawn
separately rather than as one line.

In [ ]:
official_history = historical_official_funding(ppd, plan_metrics["ppid"].dropna().astype(int).tolist())
fig, ax = plt.subplots(figsize=(10, 5))
# Upper bound taken from the data rather than hardcoded: the PPD download in
# use reaches a later fiscal year than the 2023 that was pinned here before,
# and a hardcoded bound silently drops whatever the newer file added.
_last_fy = int(official_history["fy"].max())
plot_history = official_history.loc[official_history["fy"].between(2002, _last_fy)]
ax.plot(plot_history["fy"], plot_history["equal_weighted"], marker="o", label="Selected run, equal-weighted")
ax.plot(plot_history["fy"], plot_history["liability_weighted"], marker="o", label="Selected run, liability-weighted")
ax.axhline(1.0, color="0.4", linestyle="--", linewidth=1)
ax.set_title(f"Historical official funded ratios, 2002-{_last_fy}")
ax.set_xlabel("Fiscal year")
ax.set_ylabel("Assets / liabilities")
ax.legend()
plt.show()

display(plot_history.tail(10).round(3))

In [ ]:
aaa_series = try_fetch_fred("AAA")

if aaa_series is None:
    print("AAA section skipped (no FRED data). Set FRED_API_KEY and rerun this cell.")
    aaa_pv = None
else:
    def base_year_average_rate(series, series_id, base_year):
        in_base_year = series.loc[series["date"].dt.year == base_year, series_id].dropna()
        if not in_base_year.empty:
            return float(in_base_year.mean()) / 100, f"calendar-year {base_year} average"
        through = series.loc[series["date"] <= pd.Timestamp(base_year, 12, 31), ["date", series_id]].dropna()
        if through.empty:
            raise ValueError(f"No {series_id} observations are available through {base_year}.")
        last = through.iloc[-1]
        return float(last[series_id]) / 100, f"last observation through {base_year}: {last['date'].date()}"

    def present_value_cashflows(cashflows, annual_rate):
        cashflows = np.asarray(cashflows, dtype="float64")
        periods = np.arange(1, cashflows.size + 1, dtype="float64")
        valid = np.isfinite(cashflows) & (cashflows > 0)
        return float(np.sum(cashflows[valid] / ((1 + annual_rate) ** periods[valid])))

    def aaa_cashflow_pv_summary(results, plan_metrics, aaa_rate):
        metric_lookup = plan_metrics.set_index("plan")
        rows = []
        for plan, result in sorted(results.items()):
            cashflows = result.matrix("cash_outflows").iloc[:, 0].to_numpy(dtype="float64")
            base_assets = float(result.matrix("Assets").iloc[0, 0])
            model_aal = float(result.matrix("AAL").iloc[0, 0])
            plan_discount_rate = float(result.scalars.get("discountrate", np.nan))
            aaa_pv_value = present_value_cashflows(cashflows, aaa_rate)
            rows.append({
                "plan": plan,
                "PlanName": metric_lookup.loc[plan, "PlanName"] if plan in metric_lookup.index else np.nan,
                "official_funded_ratio": metric_lookup.loc[plan, "official_funded_ratio"] if plan in metric_lookup.index else np.nan,
                "official_liability_billion": metric_lookup.loc[plan, "liability_billion"] if plan in metric_lookup.index else np.nan,
                "opening_assets_billion": base_assets / 1_000_000_000,
                "model_aal_billion": model_aal / 1_000_000_000,
                "aaa_cashflow_pv_billion": aaa_pv_value / 1_000_000_000,
                "plan_discount_rate": plan_discount_rate,
                "model_aal_funded_ratio": base_assets / model_aal if model_aal > 0 else np.nan,
                "aaa_cashflow_pv_funded_ratio": base_assets / aaa_pv_value if aaa_pv_value > 0 else np.nan,
            })
        return pd.DataFrame(rows)

    aaa_rate, aaa_rate_basis = base_year_average_rate(aaa_series, "AAA", BASE_YEAR)
    aaa_pv = aaa_cashflow_pv_summary(results, plan_metrics, aaa_rate)

    aggregate_aaa = pd.DataFrame([
        {"measure": "Official GASB funded ratio",
         "assets_billion": num_col(plan_metrics, "ActAssets_GASB").sum() / 1_000_000,
         "denominator_billion": num_col(plan_metrics, "ActLiabilities_GASB").sum() / 1_000_000,
         "funded_ratio": num_col(plan_metrics, "ActAssets_GASB").sum() / num_col(plan_metrics, "ActLiabilities_GASB").sum()},
        {"measure": "Model AAL funded ratio",
         "assets_billion": aaa_pv["opening_assets_billion"].sum(),
         "denominator_billion": aaa_pv["model_aal_billion"].sum(),
         "funded_ratio": aaa_pv["opening_assets_billion"].sum() / aaa_pv["model_aal_billion"].sum()},
        {"measure": "AAA cash-flow PV funded ratio",
         "assets_billion": aaa_pv["opening_assets_billion"].sum(),
         "denominator_billion": aaa_pv["aaa_cashflow_pv_billion"].sum(),
         "funded_ratio": aaa_pv["opening_assets_billion"].sum() / aaa_pv["aaa_cashflow_pv_billion"].sum()},
    ])
    print(f"AAA rate source: FRED AAA, {aaa_rate_basis}; rate used = {aaa_rate:.3%}")
    display(aggregate_aaa.round(3))

    plot_df = aaa_pv.dropna(subset=["official_funded_ratio", "aaa_cashflow_pv_funded_ratio", "official_liability_billion"])
    fig, ax = plt.subplots(figsize=(10, 6))
    sizes = 35 + 6 * np.sqrt(plot_df["official_liability_billion"].clip(lower=0))
    ax.scatter(plot_df["official_funded_ratio"], plot_df["aaa_cashflow_pv_funded_ratio"], s=sizes, alpha=0.65)
    limit = max(plot_df["official_funded_ratio"].max(), plot_df["aaa_cashflow_pv_funded_ratio"].max()) + 0.05
    ax.plot([0, limit], [0, limit], color="0.4", linestyle="--", linewidth=1)
    for _, row in plot_df.assign(gap=lambda x: x["official_funded_ratio"] - x["aaa_cashflow_pv_funded_ratio"]).sort_values("gap", ascending=False).head(8).iterrows():
        ax.annotate(row["plan"], (row["official_funded_ratio"], row["aaa_cashflow_pv_funded_ratio"]), fontsize=9, xytext=(4, 4), textcoords="offset points")
    ax.set_title("Official funded ratio vs AAA cash-flow PV funded ratio")
    ax.set_xlabel(f"Official GASB funded ratio, {BASE_YEAR}")
    ax.set_ylabel("Opening assets / AAA-discounted projected benefit payments")
    ax.set_xlim(0, limit)
    ax.set_ylim(0, limit)
    plt.show()

    display(
        aaa_pv[["plan", "PlanName", "official_funded_ratio", "model_aal_funded_ratio",
                "aaa_cashflow_pv_funded_ratio", "opening_assets_billion", "model_aal_billion",
                "aaa_cashflow_pv_billion", "plan_discount_rate"]]
        .sort_values("aaa_cashflow_pv_funded_ratio")
        .round(3)
    )

In [ ]:
# Their published values, for side-by-side reading rather than description.
_pkg = ROOT / "Data" / "Sources" / "brookings_bpea2021_replication" / "Main paper" / "Data & Programs"

_f1 = _pkg / "Figure 1" / "figure1_data.csv"
if _f1.exists():
    theirs = pd.read_csv(_f1)
    fig, ax = plt.subplots(figsize=(7.5, 4.2))
    ax.plot(theirs["year"], theirs["Funding_ratio"] * 100, "o-", color="#C44E52",
            label="Lenney et al. Figure 1 (AAA-discounted, Financial Accounts)")
    ax.set_xlabel("year"); ax.set_ylabel("percent")
    ax.set_title("Their Figure 1, as published")
    ax.legend(fontsize=8); plt.tight_layout(); plt.show()
else:
    print(f"not found: {_f1}")

_t1 = _pkg / "Table 1" / "table1_data.xlsx"
if _t1.exists():
    print()
    print("Their Table 1, as published:")
    display(pd.read_excel(_t1))
else:
    print(f"not found: {_t1}")

## Figures 11 and 12 — What the post-2007 reforms bought

Theirs asks: had recent reforms not happened, how much more would contributions have had
to rise to stabilise? They report it per plan, split between benefit reforms and
contribution changes since 2007.

Ours asks the same question against our own target. The counterfactual is run
`20260910_1`, in which every tier starting after 1 January 2007 takes the benefit rules of
the newest pre-2007 tier — benefit factor, COLA, vesting, salary averaging, cap and the
threshold age. Tier start dates are untouched, so the same members sit in the same tiers
and only the promise they accrue under changes.

**One half of their split is not reproduced.** They separate benefit reforms from
contribution-rate changes since 2007. Our contribution rate is a single base-year scalar
with no history, so there is nothing to reverse; only the benefit half is shown. That is a
model limitation (register E7), not a choice.

In [ ]:
# The same inversion as table 2, run on both baselines and differenced.
def inversion_on(cum, grid, offset, target, start=0):
    deltas = sorted({d for d, _ in grid})
    keys = [(d, start if d > 0 else 0) for d in deltas]
    R = np.column_stack([cum[k][:, offset] for k in keys])
    return np.array([required_increase(np.array(deltas, float), R[i], target)
                     for i in range(R.shape[0])])

base_req = inversion_on(CUM, CONTRIB_GRID, HORIZON_SHORT, TARGET_ONE)
nr_req = inversion_on(NR_CUM, NR_GRID, HORIZON_SHORT, TARGET_ONE)

reform = pd.DataFrame({"plan": GRID_PLANS, "liability_bn": LIAB,
                       "baseline": base_req, "no_reform": nr_req})
# Positive = reforms lowered what the plan must now contribute.
reform["saved_pp"] = reform["no_reform"] - reform["baseline"]
both = np.isfinite(reform["baseline"]) & np.isfinite(reform["no_reform"])

print(f"Required increase for P(shortfall by {BASE_YEAR + HORIZON_SHORT}) < {TARGET_ONE:.0%}, "
      f"baseline vs no-reform")
print(f"  plans where both are reachable, so the difference is defined: {int(both.sum())} of {len(reform)}")
print(f"  reforms lowered the requirement for {int((reform.loc[both, 'saved_pp'] > 0.01).sum())} plans, "
      f"raised it for {int((reform.loc[both, 'saved_pp'] < -0.01).sum())}")
print(f"  median saving {reform.loc[both, 'saved_pp'].median():.2f}pp, "
      f"liability-weighted {np.average(reform.loc[both, 'saved_pp'], weights=reform.loc[both, 'liability_bn']):.2f}pp")
only_nr = np.isfinite(reform["baseline"]) & ~np.isfinite(reform["no_reform"])
if only_nr.any():
    print(f"  reachable only because of the reforms: {', '.join(reform.loc[only_nr, 'plan'])}")

d = reform[both].sort_values("saved_pp")
fig, ax = plt.subplots(figsize=(11, 5.2))
ax.bar(d["plan"], d["saved_pp"], color=np.where(d["saved_pp"] > 0, "#55A868", "#C44E52"))
ax.axhline(0, color="0.3", lw=0.8)
ax.set_ylabel("percentage points of payroll saved")
ax.set_title("Figure 11 — Effect of post-2007 benefit reforms on the required contribution")
ax.tick_params(axis="x", rotation=90, labelsize=7)
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(8.2, 5.2))
ax.scatter(d["saved_pp"], d["baseline"], s=np.sqrt(d["liability_bn"]) * 4,
           alpha=0.75, color="#4C72B0")
for _, r in d.iterrows():
    if r["liability_bn"] > 120 or abs(r["saved_pp"]) > 2:
        ax.annotate(r["plan"], (r["saved_pp"], r["baseline"]), fontsize=7,
                    xytext=(4, 3), textcoords="offset points")
if len(d) > 2:
    b, a = np.polyfit(d["saved_pp"], d["baseline"], 1)
    xs = np.linspace(d["saved_pp"].min(), d["saved_pp"].max(), 40)
    ax.plot(xs, a + b * xs, ls="--", color="0.35", lw=1.2, label=f"slope {b:.2f}")
    ax.legend(fontsize=8)
ax.set_xlabel("percentage points saved by the reforms")
ax.set_ylabel("increase still required today")
ax.set_title("Figure 12 — Plans that reformed most against what they still need")
plt.tight_layout(); plt.show()
print("Marker area is proportional to accrued liability.")

## Figure 4 — Benefit payments under COLA counterfactuals

Theirs plots aggregate benefit payments as a share of GDP under the baseline, with COLAs
switched off, and with COLA set to inflation. The V6 list marks it as needing no
modification, so this reproduces the same comparison on our 40 plans.

Two counterfactual runs supply it: `20260910_2` sets every tier's COLA to zero, and
`20260910_3` sets it to each plan's own inflation assumption, which ranges 2.00% to 3.08%
against a baseline COLA median of 1.00%.

Payments are shown in dollars and, where a GDP series is available, as a share of GDP.
Ours covers these 40 plans to 2057; theirs covers the whole US state and local system to
2117, so the levels are not comparable and only the shape is.

In [ ]:
lbl = {"baseline": "baseline", "cola0": "COLA = 0", "colainf": "COLA = inflation"}
col = {"baseline": "#4C72B0", "cola0": "#55A868", "colainf": "#C44E52"}
yrs = BASE_YEAR + np.arange(CF_SERIES["baseline"]["benefit_payments"].shape[1])

# try_fetch_fred returns a DataFrame with `date` and `GDP` columns, GDP in $bn.
gdp_df = try_fetch_fred("GDP")
have_gdp = gdp_df is not None

fig, axes = plt.subplots(1, 2 if have_gdp else 1,
                         figsize=(13 if have_gdp else 7.5, 4.6), squeeze=False)
for k in ("baseline", "cola0", "colainf"):
    pay = CF_SERIES[k]["benefit_payments"].sum(axis=0)
    axes[0][0].plot(yrs, pay / 1e9, lw=2, color=col[k], label=lbl[k])
axes[0][0].set_ylabel("$bn per year")
axes[0][0].set_xlabel("year")
axes[0][0].set_title("Aggregate benefit payments, 40 plans")
axes[0][0].legend(fontsize=8)

if have_gdp:
    g = gdp_df.copy()
    g["year"] = g["date"].dt.year
    annual = g.groupby("year", as_index=False)["GDP"].mean()
    observed = annual.loc[annual["year"] <= BASE_YEAR]
    base_gdp_bn = float(observed["GDP"].iloc[-1])
    base_gdp_year = int(observed["year"].iloc[-1])

    # Same convention as the aggregate-burden section: nominal GDP grown at the
    # model's own inflation compounded with its workforce-growth assumption.
    infl = float(np.nanmean([float(r.scalars.get("Inflation", np.nan)) for r in results.values()]))
    growth = (1 + infl) * (1 + POP_GROWTH) - 1
    gdp_path_bn = base_gdp_bn * (1 + growth) ** (yrs.astype(float) - base_gdp_year)

    for k in ("baseline", "cola0", "colainf"):
        pay_bn = CF_SERIES[k]["benefit_payments"].sum(axis=0) / 1e9
        axes[0][1].plot(yrs, 100 * pay_bn / gdp_path_bn, lw=2, color=col[k], label=lbl[k])
    axes[0][1].set_ylabel("percent of GDP")
    axes[0][1].set_xlabel("year")
    axes[0][1].set_title(f"As a share of GDP (grown {growth:.2%} a year from {base_gdp_year})")
    axes[0][1].legend(fontsize=8)
else:
    print("GDP panel skipped: no FRED data. Set FRED_API_KEY and rerun.")

plt.tight_layout()
plt.show()

print(f"{'year':>6}" + "".join(f"{lbl[k]:>20}" for k in ("baseline", "cola0", "colainf")))
for t in (0, 10, 20, len(yrs) - 1):
    row = f"{yrs[t]:>6}"
    for k in ("baseline", "cola0", "colainf"):
        row += f"{CF_SERIES[k]['benefit_payments'][:, t].sum() / 1e9:>17.1f}bn"
    print(row)

_b = CF_SERIES["baseline"]["benefit_payments"].sum(axis=0)[-1]
for k in ("cola0", "colainf"):
    _v = CF_SERIES[k]["benefit_payments"].sum(axis=0)[-1]
    print(f"  {lbl[k]:<18} {yrs[-1]} payments are {(_v / _b - 1) * 100:+.1f}% against baseline")

if have_gdp:
    print("\nThe GDP path is a constant-growth projection off the last observed level, not a "
          "forecast. It makes the three scenarios comparable to each other; it does not make "
          "the levels comparable to Lenney figure 4, which covers the whole US system to 2117.")

## What these exhibits do not cover

Every item on the V6 list is now implemented above. Three limitations are worth carrying
into any reading of them, all of them properties of the model rather than of the figures.

**The contribution half of figures 11 and 12 is missing.** They split the effect of
post-2007 policy between benefit reforms and contribution-rate changes. Our contribution
rate is one base-year scalar held constant for the whole projection, so there is no
history to reverse and only the benefit half is shown (register E7).

**The horizon is 2057.** Their exhaustion buckets run past 75 years and include a "never"
category; ours cannot. Where a plan does not reach a target inside the grid it is reported
as not reachable rather than dropped, but that is bounded by the projection as well as by
the +20pp ceiling.

**The early-retirement reduction rests largely on an assumption.** Of the 128 plan-tier
rates, 16 are unambiguous extracted values, 15 are a weaker plan-level inference and 97
are the assumed 6%. The `source` column of
`Data/Common/states/early_retirement_reduction.csv` says which is which, and extracting
the rest is a job for the `Data Extraction/` pipeline.

---

# Archive

Everything below is the analysis as it stood before this reorganisation. It is
kept, not deleted, and it is sorted by how useful it is for the exhibits above
rather than by what it was originally grouped under. Headings inside each bucket
are the original section headings, pushed down two levels so they nest here.

The infrastructure at the top of this notebook (run inventory, output loading,
metric construction) is untouched, so every cell below still runs in place.

## Useful

Sections that already fill, or directly feed, one of the Lenney exhibits above.

---
### Stochastic Risk Metrics

These sections use the full simulated distribution rather than central tendencies. Asset exhaustion ("default" in uncovered-benefit terms) is defined per path as the first projected year in which a plan's assets reach zero; after exhaustion, benefits exceed dedicated funding and the sponsor must pay benefits from current revenue.

#### Asset Exhaustion Timing By Liability Exposure

The chart shows, for each projected year, the amount of liability expected to run out of assets **for the first time in that year**: each plan's base-year GASB liability multiplied by the simulated probability that it first exhausts then, summed across plans.

It is drawn one bar per year rather than in bins. Bins would have to be unequal — the projection ends at 2056, so a fourth ten-year bin cannot fit — and unequal bins distort the shape badly here: the final four-year window carries the *highest* per-year amount of any period while showing a shorter bar than the decade before it. The liability that never runs out of assets is drawn as the blue bar at the right, set apart by a gap because it is a category rather than a year. It is about forty times the tallest yearly bar, so the vertical axis is broken: without the break every yearly bar would be flattened to a few percent of the plot height.

The binned table below is kept for reference and carries a per-year column, which is the column to compare across rows.

In [ ]:
liability_bins = liability_weighted_exhaustion_bins(results, plan_metrics)
by_year, never_billion, total_billion = liability_weighted_exhaustion_by_year(results, plan_metrics)

# Broken y-axis. The never-exhausting bar is about forty times the tallest yearly
# bar, so on one continuous scale every yearly bar collapses to a couple of
# percent of the plot height and the profile becomes unreadable. The break keeps
# both in the same chart against a labelled scale. For a single continuous axis
# instead, drop ax_top and plot everything on one Axes.
fig, (ax_top, ax_bot) = plt.subplots(
    2, 1, sharex=True, figsize=(11, 6.5),
    gridspec_kw={"height_ratios": [1, 3], "hspace": 0.06})

x_years = by_year["year"].to_numpy(dtype="float64")
h_years = by_year["expected_liability_billion"].to_numpy(dtype="float64")
x_never = float(ASSET_LAST_YEAR + 2)          # a gap, so it reads as a separate category

for ax in (ax_top, ax_bot):
    ax.bar(x_years, h_years, width=0.8, color="tab:red", alpha=0.75,
           label="Exhausts this year")
    ax.bar([x_never], [never_billion], width=0.8, color="tab:blue", alpha=0.85,
           label="Never exhausts")
    ax.xaxis.grid(False)

ax_top.set_ylim(never_billion * 0.94, never_billion * 1.06)
ax_bot.set_ylim(0, float(h_years.max()) * 1.18)

ax_top.spines["bottom"].set_visible(False)
ax_bot.spines["top"].set_visible(False)
ax_top.tick_params(bottom=False, labelbottom=False)

# Diagonal marks showing the axis is broken
_break = dict(marker=[(-1, -0.6), (1, 0.6)], markersize=8, linestyle="none",
              color="0.45", mec="0.45", mew=1, clip_on=False)
ax_top.plot([0, 1], [0, 0], transform=ax_top.transAxes, **_break)
ax_bot.plot([0, 1], [1, 1], transform=ax_bot.transAxes, **_break)

ax_top.annotate(f"${never_billion:,.0f}bn\n{never_billion / total_billion:.1%} of all liab.",
                xy=(x_never, never_billion), xytext=(0, 6), textcoords="offset points",
                ha="center", va="bottom", fontsize=9, annotation_clip=False)
_peak = int(np.argmax(h_years))
ax_bot.annotate(f"${h_years[_peak]:,.0f}bn", xy=(x_years[_peak], h_years[_peak]),
                xytext=(0, 4), textcoords="offset points", ha="center", fontsize=8)

ticks = [y for y in x_years if int(y) % 5 == 0]
ax_bot.set_xticks(ticks + [x_never])
ax_bot.set_xticklabels([f"{int(y)}" for y in ticks] + [f"Never\nby {ASSET_LAST_YEAR}"])
ax_bot.set_xlabel("Fiscal year in which assets first reach zero")
ax_bot.set_ylabel("Liability at inception, expected ($bn)")
ax_top.set_title("Liability of plans exhausting, by year")
ax_top.legend(loc="center left", fontsize=9, frameon=False)
plt.show()

# The binned view is kept for reference, with the per-year column that makes the
# four rows comparable -- the bins are not the same width.
display(liability_bins.round(3))


#### Asset Exhaustion By Liability Exposure, Cumulative

The same quantity as the chart above, accumulated instead of taken year by year:
how much liability has run out of assets **by** each fiscal year, rather than how
much runs out **in** it. A plan counted once in the bar above appears in every
bar from that year onward here.

Two things worth knowing about it. It contains no information the previous chart
does not — it is that chart's running total — and it is also the liability-weighted
curve from the exhaustion-CDF figure below, expressed in dollars rather than as a
probability. It is here because a cumulative amount is the more natural way to ask
how much is at stake by a given date.

Unlike the chart above, this one needs no broken axis: the final red bar and the
blue bar are the two parts of the same total, so they are of comparable size and
together account for every dollar of liability in the study.

**A plan that runs out of assets does not necessarily stay out.** Assets at
zero return positive in any year where contributions exceed benefit payments.
Across the 40 plans this happens for **IL33 alone**, on every one of the 1,851
paths where it runs out: it reaches zero between 2033 and 2056 and climbs back
out in 2056-57, the only two years in which its contributions exceed its benefit
outflows. So "has run out by this year" and "is out in this year" are not the
same series, and this chart measures the first of the two.


In [ ]:
cumulative, never_cum_billion, total_cum_billion = liability_weighted_exhausted_by_year(
    results, plan_metrics)

fig, ax = plt.subplots(figsize=(11, 5.5))

x_years_c = cumulative["year"].to_numpy(dtype="float64")
h_years_c = cumulative["expected_liability_billion"].to_numpy(dtype="float64")
x_never_c = float(ASSET_LAST_YEAR + 2)        # a gap, so it reads as a separate category

ax.bar(x_years_c, h_years_c, width=0.8, color="tab:red", alpha=0.75,
       label="Has exhausted")
ax.bar([x_never_c], [never_cum_billion], width=0.8, color="tab:blue", alpha=0.85,
       label="Never exhausts")
ax.xaxis.grid(False)

ax.annotate(f"${never_cum_billion:,.0f}bn\n{never_cum_billion / total_cum_billion:.1%}",
            xy=(x_never_c, never_cum_billion), xytext=(0, 5), textcoords="offset points",
            ha="center", va="bottom", fontsize=9)
ax.annotate(f"${h_years_c[-1]:,.0f}bn\n{h_years_c[-1] / total_cum_billion:.1%}",
            xy=(x_years_c[-1], h_years_c[-1]), xytext=(0, 5), textcoords="offset points",
            ha="center", va="bottom", fontsize=9)

ticks_c = [y for y in x_years_c if int(y) % 5 == 0]
ax.set_xticks(ticks_c + [x_never_c])
ax.set_xticklabels([f"{int(y)}" for y in ticks_c] + [f"Never\nby {ASSET_LAST_YEAR}"])
ax.set_xlabel("Fiscal year")
ax.set_ylabel("Liability at inception, expected ($bn)")
ax.set_title("Cumulative liability of plans exhausted")
ax.set_ylim(0, max(float(h_years_c.max()), never_cum_billion) * 1.20)
ax.legend(loc="upper left", fontsize=9, frameon=False)
plt.show()

print(f"By {ASSET_LAST_YEAR}: ${h_years_c[-1]:,.0f}bn of liability expected to have "
      f"exhausted assets and ${never_cum_billion:,.0f}bn expected not to, "
      f"of ${total_cum_billion:,.0f}bn total.")

#### Exhaustion-Year Distribution

The first figure shows the cumulative probability that assets are exhausted by each projection year: the liability-weighted curve across all plans, plus the individual curves for the highest-risk plans. The second figure shows each plan's full exhaustion CDF as small multiples of risk, sorted from highest to lowest 35-year exhaustion probability.

In [ ]:
cdf_by_plan, cdf_weighted = exhaustion_cdf(results, plan_metrics, max_year=MAX_OFFSET)

top_risk = exhaustion.nlargest(6, EXH_COL_MAX)["plan"].tolist()
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(BASE_YEAR + cdf_weighted.index, cdf_weighted * 100, color="black", linewidth=2.5, label="Liability-weighted, all plans")
for plan in top_risk:
    ax.plot(BASE_YEAR + cdf_by_plan.index, cdf_by_plan[plan] * 100, linewidth=1, label=plan)
ax.set_title("Cumulative probability of asset exhaustion by projection year")
ax.set_xlabel("Fiscal year")
ax.set_ylabel("P(exhausted by year) (%)")
ax.set_ylim(0, 100)
ax.legend()
plt.show()

# Exhaustion CDF table at selected horizons (plan rows sorted by risk)
horizon_cols = [5, 10, 15, 20, 25, 30, MAX_OFFSET]
cdf_table = (cdf_by_plan.loc[horizon_cols].T * 100).round(1)
cdf_table.columns = [f"by_{BASE_YEAR + h}" for h in horizon_cols]
display(cdf_table.sort_values(f"by_{ASSET_LAST_YEAR}", ascending=False))

#### Plan-Level Exhaustion Probabilities

Each plan's cumulative probability of exhausting assets by three fiscal years. The bars nest because the measure is cumulative: a plan that has exhausted by 2032 has also exhausted by 2042. The full per-plan frame is available as `exhaustion` for custom queries.

**A severity panel used to sit beside this and has been removed.** It plotted `mean_years_insolvent`, the average number of projection years spent with zero assets — averaged over *every* path, including those where the plan never runs out and which therefore contribute zero. That makes it roughly the exhaustion probability multiplied by the duration, so it largely restated the bars on the left rather than adding a second dimension, and its title ("horizon") described a point in time while the quantity was a count of years. The column is still computed and available in `exhaustion` if wanted.

**Severity now lives in its own section below**, where it is measured *conditionally* — given that a plan does run out, how large the shortfall is and how long it lasts — which separates the size of the problem from how often it happens.


In [ ]:
t = exhaustion.sort_values(EXH_COL_MAX, ascending=True).reset_index(drop=True)
y = np.arange(len(t))

fig, ax = plt.subplots(figsize=(8, max(6, len(t) * 0.30)))
ax.barh(y, t[EXH_COL_MAX] * 100, color="#f4b8b0", label=f"by {ASSET_LAST_YEAR}")
ax.barh(y, t[EXH_COLS[SCATTER_HORIZON]] * 100, color="#d65f5f", label=f"by {BASE_YEAR + 20}")
ax.barh(y, t[EXH_COLS[10]] * 100, color="#8f1d1d", label=f"by {BASE_YEAR + 10}")
ax.set_yticks(y)
ax.set_yticklabels(t["plan"], fontsize=8)
ax.set_xlabel("P(assets exhausted) (%)")
ax.set_title("Exhaustion probability by horizon")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()


#### Probability Below Funding-Ratio Thresholds Over Time

The figure shows the cross-plan average probability that simulated funding ratios fall below selected thresholds in each projection year, over the full usable horizon. The probability of being *at or above* full funding in a year is one minus the `< 1.0` line.

In [ ]:
risk_over_time = ra.threshold_risk_over_time(results, thresholds=(0.4, 0.6, 0.8, 1.0), graph_years=N_PROJ)
display(risk_over_time.head())

fig, ax = ra.plot_threshold_risk(results, thresholds=(0.4, 0.6, 0.8, 1.0), graph_years=N_PROJ)
ax.set_title("P(funded ratio below threshold), mean across plans")
ax.set_ylabel("Probability, mean across plans")
ax.set_xlabel("Fiscal year")
plt.show()

#### Exhaustion Risk By Official Funded Ratio

Cross-sectional relationship between each plan's official funded ratio in the base year and its simulated probability of exhausting assets within 20 years. Bubble size is proportional to GASB liabilities. The fitted line is a simple linear regression used only as a visual summary.

In [ ]:
risk_scatter = exhaustion.merge(
    plan_metrics[["plan", "PlanName", "assets_liabilities", "liability_billion", "unfunded_liabilities_payroll", "contribution_rate"]],
    on="plan",
    suffixes=("", "_metric"),
)
plot_df = risk_scatter.dropna(subset=["official_funded_ratio", EXH_COLS[SCATTER_HORIZON], "liability_billion"])
fig, ax = plt.subplots(figsize=(10, 6))
sizes = 35 + 6 * np.sqrt(plot_df["liability_billion"].clip(lower=0))
ax.scatter(plot_df["official_funded_ratio"], plot_df[EXH_COLS[SCATTER_HORIZON]], s=sizes, alpha=0.65)
sns.regplot(data=plot_df, x="official_funded_ratio", y=EXH_COLS[SCATTER_HORIZON], scatter=False, ax=ax, color="black", line_kws={"linewidth": 1})
for _, row in plot_df.sort_values(EXH_COLS[SCATTER_HORIZON], ascending=False).head(8).iterrows():
    ax.annotate(row["plan"], (row["official_funded_ratio"], row[EXH_COLS[SCATTER_HORIZON]]), fontsize=9, xytext=(4, 4), textcoords="offset points")
ax.set_title(f"P(exhaustion by {BASE_YEAR + SCATTER_HORIZON}) vs reported funded ratio at inception")
ax.set_xlabel("Reported funded ratio at inception (assets / accrued liability, GASB)")
ax.set_ylabel(f"P(exhaustion by {BASE_YEAR + SCATTER_HORIZON})")
ax.set_ylim(-0.02, min(1.02, max(0.1, plot_df[EXH_COLS[SCATTER_HORIZON]].max() + 0.08)))
plt.show()

display(plot_df[["plan", "PlanName", "official_funded_ratio", EXH_COLS[10], EXH_COLS[SCATTER_HORIZON], EXH_COLS[30], "liability_billion", "unfunded_liabilities_payroll", "contribution_rate"]].sort_values(EXH_COLS[SCATTER_HORIZON], ascending=False).round(3))

---
### Cross-Plan And Aggregate Dynamics

#### Aggregate Funded Ratio And Unfunded Liability

Both figures aggregate simulated dollar balances across plans **within each Monte Carlo path** and then summarize across paths. Because all plans share common market shocks, these fans are genuine aggregate risk distributions: a bad path is bad for every plan at once. The first figure shows the aggregate funded ratio; the second the aggregate unfunded liability (AAL − assets) in dollars, which becomes negative when aggregate assets exceed aggregate liabilities. Years with nonpositive aggregate AAL (placeholder final year) are excluded.

#### Joint Failure: What Happens Across Plans Within One Market Path

Everything above measures one plan at a time. This section measures how many
plans fail **together**, which is a different question and is only answerable
because every plan in the run is simulated against the same market history:
simulation column *n* is one shared sequence of market outcomes, so a path that
is bad for one plan is bad for all of them at once.

The comparison at the end is the point of the section. Each plan's individual
exhaustion probability is a fact about that plan. Multiplying those individual
probabilities together — treating plans as if they failed independently — gives
a very different answer from what the simulation actually produces, because in
reality the plans share one economy. The gap between the two is a measure of
how much a plan-by-plan reading understates the risk of many plans failing at
the same time.

The independence benchmark is computed exactly, not simulated: it is the
Poisson-binomial distribution implied by the 40 individual probabilities.

In [ ]:
def exhaustion_indicators(results, max_offset):
    """(n_plans, max_offset, n_sim) boolean: has this plan exhausted BY this year?"""
    plans = sorted(results)
    n_sim = min(r.n_simulations for r in results.values())
    out = np.zeros((len(plans), max_offset, n_sim), dtype=bool)
    for i, plan in enumerate(plans):
        assets = results[plan].matrix("Assets").to_numpy(dtype="float64")
        out[i] = np.maximum.accumulate(assets[1:max_offset + 1, :n_sim] <= 0, axis=0)
    return plans, out


joint_plans, exhausted_by = exhaustion_indicators(results, MAX_OFFSET)
n_failed = exhausted_by.sum(axis=0)                      # (MAX_OFFSET, n_sim)
joint_years = np.arange(BASE_YEAR + 1, BASE_YEAR + MAX_OFFSET + 1)

liab = plan_metrics.set_index("plan").reindex(joint_plans)["liability_billion"].to_numpy(dtype="float64")
liab = np.where(np.isfinite(liab), liab, 0.0)
share_failed = (exhausted_by * liab[:, None, None]).sum(axis=0) / liab.sum()

# --- how many plans fail together, over time --------------------------------
count_fan = fan_table(joint_years, n_failed.astype(float))
share_fan = fan_table(joint_years, share_failed * 100)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
plot_fan(axes[0], count_fan, "Plans exhausted", color="tab:red")
axes[0].set_title("Number of plans with exhausted assets")
axes[0].set_ylabel("Plans (of %d)" % len(joint_plans))
plot_fan(axes[1], share_fan, "Liability share", color="tab:red")
axes[1].set_title("Share of total liabilities in exhausted plans")
axes[1].set_ylabel("Percent of total liabilities")
for ax in axes:
    ax.set_xlabel("Fiscal year")
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

# --- simulated joint distribution vs an independence benchmark --------------
final_counts = n_failed[-1, :]
observed = np.bincount(final_counts, minlength=len(joint_plans) + 1) / final_counts.size

marginals = exhausted_by[:, -1, :].mean(axis=1)
independent = np.zeros(len(marginals) + 1)
independent[0] = 1.0
for prob in marginals:                       # exact Poisson-binomial convolution
    shifted = np.zeros_like(independent)
    shifted[0] = independent[0] * (1 - prob)
    shifted[1:] = independent[1:] * (1 - prob) + independent[:-1] * prob
    independent = shifted

k = np.arange(len(observed))
fig, ax = plt.subplots(figsize=(11, 5.5))
ax.bar(k, observed * 100, color="tab:red", alpha=0.65, label="Simulated (shared market history)")
ax.step(k, independent * 100, where="mid", color="black", linewidth=1.8,
        label="If plans failed independently")
ax.set_title(f"How many of the {len(joint_plans)} plans have exhausted assets by {ASSET_LAST_YEAR}")
ax.set_xlabel("Number of plans with exhausted assets")
ax.set_ylabel("Probability (%)")
ax.legend()
plt.show()

tail = pd.DataFrame({
    "at_least_n_plans": k,
    "simulated": observed[::-1].cumsum()[::-1],
    "if_independent": independent[::-1].cumsum()[::-1],
})
tail["ratio"] = tail["simulated"] / tail["if_independent"].replace(0, np.nan)
print(f"Mean number of plans exhausted by {ASSET_LAST_YEAR}: "
      f"simulated {float((observed * k).sum()):.2f}, independent {float((independent * k).sum()):.2f} "
      "(these agree by construction -- only the SPREAD differs)")
display(tail.loc[tail["at_least_n_plans"].isin([5, 10, 15, 20, 25, 30])].round(4))

# --- do the largest plans fail together? ------------------------------------
big = plan_metrics.nlargest(3, "liability_billion")["plan"].tolist()
rows = []
idx = [joint_plans.index(b) for b in big]
together = exhausted_by[idx, -1, :].all(axis=0).mean()
product = float(np.prod([marginals[i] for i in idx]))
rows.append({"group": " + ".join(big), "simulated_joint": together,
             "product_of_marginals": product,
             "ratio": together / product if product > 0 else np.nan})
display(pd.DataFrame(rows).round(4))

In [ ]:
ratio_paths = np.divide(agg_assets, agg_aal, out=np.full_like(agg_assets, np.nan), where=agg_aal > 0)
ratio_fan = fan_table(agg_years, ratio_paths)
unfunded_fan = fan_table(agg_years, (agg_aal - agg_assets) / 1_000_000_000)

fig, ax = plt.subplots(figsize=(10, 5))
plot_fan(ax, ratio_fan, "Aggregate funded ratio")
ax.axhline(1.0, color="0.4", linestyle="--", linewidth=1)
ax.set_title("Aggregate funded-ratio dynamics (common market shocks)")
ax.set_xlabel("Fiscal year")
ax.set_ylabel("Assets / AAL")
ax.legend()
plt.show()

fig, ax = plt.subplots(figsize=(10, 5))
plot_fan(ax, unfunded_fan, "Aggregate unfunded AAL", color="tab:red")
ax.axhline(0.0, color="0.4", linewidth=1)
ax.set_title("Aggregate unfunded liability dynamics")
ax.set_xlabel("Fiscal year")
ax.set_ylabel("Billions of dollars")
ax.legend()
plt.show()

display(ratio_fan.head(10).round(3))

#### GDP-Normalized Aggregate Burden

This section normalizes the aggregate balance sheet by nominal GDP. Observed nominal GDP from the official FRED API is used only through the model base year; afterwards GDP grows at a model-consistent nominal rate (average model inflation compounded with the 1 percent population-growth assumption). This is a deterministic scaling denominator, not a macro forecast, so all dispersion in the fan comes from the simulated numerator.

The section requires `FRED_API_KEY` (notebook variable or environment variable); without it, it prints a skip message and the rest of the notebook still runs.

In [ ]:
gdp_series = try_fetch_fred("GDP")

if gdp_series is None:
    print("GDP normalization skipped (no FRED data). Set FRED_API_KEY and rerun this cell.")
else:
    gdp_series["year"] = gdp_series["date"].dt.year
    gdp_annual = gdp_series.groupby("year", as_index=False)["GDP"].mean()
    gdp_observed = gdp_annual.loc[gdp_annual["year"] <= BASE_YEAR]
    base_row = gdp_observed.loc[gdp_observed["year"] == BASE_YEAR]
    if base_row.empty:
        base_row = gdp_observed.tail(1)
    base_gdp_billion = float(base_row["GDP"].iloc[-1])
    base_gdp_year = int(base_row["year"].iloc[-1])

    model_inflation = float(np.nanmean([float(r.scalars.get("Inflation", np.nan)) for r in results.values()]))
    model_nominal_gdp_growth = (1 + model_inflation) * (1 + MODEL_POPULATION_GROWTH) - 1

    gdp_path = base_gdp_billion * (1 + model_nominal_gdp_growth) ** (agg_years.astype(float) - base_gdp_year)

    display(pd.DataFrame([
        {"item": "Base GDP year", "value": base_gdp_year},
        {"item": "Base nominal GDP, billions", "value": base_gdp_billion},
        {"item": "Average model inflation", "value": model_inflation},
        {"item": "Projected nominal GDP growth", "value": model_nominal_gdp_growth},
    ]))

    unfunded_pct_paths = 100 * (agg_aal - agg_assets) / 1_000_000_000 / gdp_path[:, None]
    unfunded_pct_fan = fan_table(agg_years, unfunded_pct_paths)

    fig, ax = plt.subplots(figsize=(10, 5))
    plot_fan(ax, unfunded_pct_fan, "Unfunded AAL / GDP", color="tab:red")
    ax.axhline(0.0, color="0.4", linewidth=1)
    ax.set_title("Aggregate unfunded liability as a share of projected GDP")
    ax.set_xlabel("Fiscal year")
    ax.set_ylabel("Percent of GDP")
    ax.legend()
    plt.show()

    assets_pct_mean = 100 * np.nanmean(agg_assets, axis=1) / 1_000_000_000 / gdp_path
    aal_pct_mean = 100 * np.nanmean(agg_aal, axis=1) / 1_000_000_000 / gdp_path
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(agg_years, assets_pct_mean, label="Assets (mean)")
    ax.plot(agg_years, aal_pct_mean, label="AAL (mean)")
    ax.axhline(0, color="0.4", linewidth=1)
    ax.set_title("Aggregate assets and AAL as a share of projected GDP (means)")
    ax.set_xlabel("Fiscal year")
    ax.set_ylabel("Percent of GDP")
    ax.legend()
    plt.show()

    display(unfunded_pct_fan.tail(5).round(3))

---
### Contribution Sensitivity

Every figure above describes the plans as they are. This part asks what changes if
**more money is paid in**, and by how much.

**What the scenario is.** A permanent increase in contributions, expressed in
percentage points of payroll, starting immediately and continuing every year of the
projection. An increase of 2.5 means the plan receives an extra 2.5% of its payroll
each year on top of the employee and employer contributions it already collects.
Because it is a share of payroll, the extra money grows with the projected workforce
and wages rather than being a fixed sum.

**What it does not change.** The liability side is untouched: the same benefits are
promised, to the same people, on the same schedule. Verified directly — the accrued
liability path is bit-identical across every point of the grid. So this measures the
effect of *funding the existing promise more heavily*, not of changing the promise.

**Three properties of the scenario worth knowing.**

- The increase is paid **even in years when the plan is more than fully funded**.
  The ordinary contribution stops above full funding under the model's own rule; the
  add-on does not. That keeps the lever meaning the same thing on every path, at the
  cost of charging money in states where a real sponsor might not.
- It starts **in the first projected year**. Delayed starts are a separate grid that
  has not been run.
- Every scenario shares the baseline's **market histories** (seed 123), so a given
  simulation column is the same sequence of good and bad years across the whole grid.
  Differences between scenarios are the policy, not the draw.

In [ ]:
import json as _json
import pickle as _pickle

# Risk targets for the inversion below. Change these freely -- they are a reporting
# choice, not a property of the model. Only 0.01 has any standing behind it: it is
# the example in the 2026-06-10 scenario design note ("minimum contribution increase
# for P(exhaust) <= 1%"). The rest are here to show the shape.
RISK_TARGETS = (0.50, 0.25, 0.10, 0.05, 0.01)


def discover_contribution_grid(root, baseline_tag):
    """Find every scenario run that adds contributions on top of `baseline_tag`.

    Discovered from the runs themselves rather than hardcoded, so adding a grid
    point means running it, not editing this cell. A run qualifies when it reuses
    this baseline's liabilities, applies its increase from year 0, and adds a
    positive amount.
    """
    runs = Path(root) / "Results" / "Runs"
    grid = {0.0: baseline_tag}
    skipped = []
    for d in sorted(runs.iterdir()):
        if not d.is_dir() or d.name.startswith("_") or d.name == baseline_tag:
            continue
        found = list(d.glob(f"*/*_parquet/scalars.parquet"))
        if not found:
            continue
        s = pd.read_parquet(found[0])
        scal = dict(zip(s["name"], s["value"]))
        raw = scal.get("scenario_json")
        if not raw:
            continue
        sc = _json.loads(raw)
        add = float(sc.get("contrib_add") or 0.0)
        if add <= 0:
            continue
        if sc.get("detal_run_tag") != baseline_tag or int(sc.get("policy_start") or 0) != 0:
            skipped.append((d.name, sc.get("detal_run_tag"), sc.get("policy_start")))
            continue
        grid[add] = d.name
    if skipped:
        print("[note] scenario runs present but excluded (different baseline or start year):")
        for name, dt, ps in skipped:
            print(f"        {name}: detal={dt}, policy_start={ps}")
    return dict(sorted(grid.items()))


def load_exhaustion_indicators(root, tag, plans, max_offset):
    """(n_plans, max_offset, n_sim) boolean: has this plan exhausted BY this year?

    Reads each plan's assets, reduces immediately to the cumulative indicator and
    lets the float matrix go. Keeping seven full grids of floats would be about
    800 MB; the indicators are roughly 14 MB per scenario.
    """
    runs = Path(root) / "Results" / "Runs" / tag
    out = None
    for i, plan in enumerate(plans):
        f = next(iter((runs / plan).rglob("Assets.parquet")))
        a = pd.read_parquet(f).to_numpy(dtype="float64")
        if out is None:
            out = np.zeros((len(plans), max_offset, a.shape[1]), dtype=bool)
        out[i] = np.maximum.accumulate(a[1:max_offset + 1, :] <= 0, axis=0)
    return out


CONTRIB_GRID = discover_contribution_grid(ROOT, RUN_TAG)
GRID_PLANS = sorted(results)
print(f"Contribution grid discovered against {RUN_TAG}:")
for delta, tag in CONTRIB_GRID.items():
    print(f"   +{delta:>5.1f}pp of payroll   {tag}")

grid_exhausted = {d: load_exhaustion_indicators(ROOT, t, GRID_PLANS, MAX_OFFSET)
                  for d, t in CONTRIB_GRID.items()}

# P(exhausted by the final projected year), plan by grid point
risk = pd.DataFrame(
    {d: {p: float(ind[i, -1, :].mean()) for i, p in enumerate(GRID_PLANS)}
     for d, ind in grid_exhausted.items()})
risk.index.name = "plan"
GRID_DELTAS = np.array(sorted(CONTRIB_GRID), dtype=float)

_liab = plan_metrics.set_index("plan").reindex(GRID_PLANS)["liability_billion"]
LIAB_W = np.where(np.isfinite(_liab.to_numpy(dtype=float)), _liab.to_numpy(dtype=float), 0.0)

print(f"\n{len(GRID_PLANS)} plans x {len(GRID_DELTAS)} grid points; "
      f"risk measured as P(exhaustion by {ASSET_LAST_YEAR}).")
_mono = risk.diff(axis=1).iloc[:, 1:]
print("Risk falls at every step for every plan:", bool((_mono <= 0).all().all()))

#### The risk curves

Each thin line is one plan: its probability of running out of assets by the end of
the projection, against how much extra contribution it receives. The heavy line is
the liability-weighted average across all plans, so it answers "what happens to the
money" rather than "what happens to the typical plan".

The curves are the raw object; everything after this is a way of reading them.
Two things to look for: how steep a plan's line is, and whether it is still falling
at the right-hand edge or has flattened out.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13.5, 5.5))

# Every plan drawn identically. Colouring a subset implies the others are a
# background the eye can ignore, which is exactly wrong when the point is how
# differently the same policy acts on different plans. Only the aggregate is
# distinguished, because it is a different kind of object.
ax = axes[0]
for p in GRID_PLANS:
    ax.plot(GRID_DELTAS, risk.loc[p] * 100, color="0.7", linewidth=0.9, zorder=1)
w = (risk.T.to_numpy() @ LIAB_W) / LIAB_W.sum()
ax.plot(GRID_DELTAS, w * 100, color="black", linewidth=3,
        label="All plans, weighted by liability at inception", zorder=4)
ax.set_xlabel("Permanent contribution increase (pp of payroll)")
ax.set_ylabel(f"P(exhaustion by {ASSET_LAST_YEAR}) (%)")
ax.set_title("P(exhaustion) by contribution increase")
ax.set_ylim(-2, 102)
ax.legend(fontsize=8, loc="lower left")

ax = axes[1]
for p in GRID_PLANS:
    base = risk.loc[p, 0.0]
    if base <= 0.01:
        continue
    ax.plot(GRID_DELTAS, risk.loc[p] / base * 100, color="0.7", linewidth=0.9, zorder=1)
_wn = w / w[0] * 100
ax.plot(GRID_DELTAS, _wn, color="black", linewidth=3,
        label="All plans, weighted by liability at inception", zorder=4)
ax.set_xlabel("Permanent contribution increase (pp of payroll)")
ax.set_ylabel("P(exhaustion), % of level at 0pp")
ax.set_title("P(exhaustion) relative to baseline")
ax.legend(fontsize=8, loc="lower left")
ax.set_ylim(0, 105)
plt.tight_layout()
plt.show()

display(risk.loc[risk[0.0].sort_values(ascending=False).index].head(15).round(4))


#### The inversion: what it costs to reach a target

The curves read forwards — "given this much extra contribution, what is the risk".
The inversion reads them backwards, which is the question a sponsor actually asks:
**"how much extra contribution do I need to get risk down to X?"**

For each plan and each target, the smallest increase on the grid that reaches it,
interpolated linearly between the two grid points that straddle the crossing. Plans
already below a target at zero cost nothing; plans that never reach it within the
grid are reported as beyond its range rather than extrapolated, because the curves
bend and extrapolating them would invent a number.

**The targets are a reporting choice, not a model property.** Change `RISK_TARGETS`
in the cell above. Only the 1% level has any standing behind it, as the example in
the scenario design note.

In [ ]:
def required_increase(row, target, deltas):
    """Smallest increase reaching P(exhaust) <= target; NaN if outside the grid."""
    y = row.to_numpy(dtype=float)
    if y[0] <= target:
        return 0.0
    for k in range(1, len(deltas)):
        if y[k] <= target:
            y0, y1 = y[k - 1], y[k]
            if y0 == y1:
                return deltas[k]
            return deltas[k - 1] + (y0 - target) * (deltas[k] - deltas[k - 1]) / (y0 - y1)
    return np.nan


inversion = pd.DataFrame(
    {f"to_{t:g}": risk.apply(required_increase, axis=1, target=t, deltas=GRID_DELTAS)
     for t in RISK_TARGETS})
inversion.insert(0, "risk_at_0pp", risk[0.0])
inversion = inversion.loc[inversion.risk_at_0pp.sort_values(ascending=False).index]

fig, axes = plt.subplots(1, 2, figsize=(13.5, max(5.5, len(GRID_PLANS) * 0.22)))

ax = axes[0]
target_main = RISK_TARGETS[len(RISK_TARGETS) // 2]
col = f"to_{target_main:g}"
# Plans already at or below the level are kept, at zero, rather than dropped:
# excluding them would make the sample silently incomplete.
# Plans needing more than the grid covers are shown at the top with a bar one
# step longer than the largest tested increase, labelled with a "+", so the
# sample is complete rather than silently truncated.
_over = GRID_DELTAS[-1] * 1.12
t = inversion[[col]].copy()
t["plot_value"] = t[col].fillna(_over)
t = t.sort_values("plot_value")
_beyond = t[col].isna().to_numpy()
ax.barh(np.arange(len(t)), t["plot_value"],
        color=np.where(_beyond, "tab:red", "tab:green"), alpha=0.8)
for _i in np.flatnonzero(_beyond):
    ax.annotate(f"{GRID_DELTAS[-1]:g}+", (_over, _i), xytext=(3, 0),
                textcoords="offset points", fontsize=7, va="center")
ax.set_yticks(np.arange(len(t)))
ax.set_yticklabels(t.index, fontsize=8)
ax.set_xlabel("Permanent contribution increase needed (pp of payroll)")
ax.set_title(f"Contribution increase to reach P(exhaustion) {target_main:.0%}")

ax = axes[1]
for t_ in RISK_TARGETS:
    c = inversion[f"to_{t_:g}"].dropna()
    c = c[c > 0]
    if len(c) == 0:
        continue
    xs = np.sort(c.to_numpy())
    ax.step(xs, np.arange(1, len(xs) + 1) / len(GRID_PLANS) * 100, where="post",
            label=f"<= {t_:.0%}", linewidth=2)
ax.set_xlabel("Permanent contribution increase (pp of payroll)")
ax.set_ylabel("Plans reaching the level (%)")
ax.set_title("Plans reaching the level, by contribution increase")
ax.legend(fontsize=8)
ax.set_xlim(0, GRID_DELTAS[-1])
plt.tight_layout()
plt.show()

disp = inversion.copy()
for c in disp.columns[1:]:
    disp[c] = disp[c].map(lambda v: "already there" if v == 0 else
                          (f"over {GRID_DELTAS[-1]:g}" if not np.isfinite(v) else f"{v:.2f}"))
disp["risk_at_0pp"] = disp["risk_at_0pp"].map(lambda v: f"{v:.4f}")
display(disp)

print("Coverage of the grid, by target:")
for t_ in RISK_TARGETS:
    c = inversion[f"to_{t_:g}"]
    print(f"   <= {t_:>5.0%}   already there {int((c == 0).sum()):2d}   "
          f"reached within {GRID_DELTAS[-1]:g}pp {int(((c > 0) & np.isfinite(c)).sum()):2d}   "
          f"beyond the grid {int(c.isna().sum()):2d}")

#### What a given increase buys, across the whole study

The inversion is per plan. This is the same information read across plans at once:
for any contribution increase, what share of the study sits under a given risk
target. Two versions, and the difference between them matters.

**By plan** counts each plan equally. **By liability** weights each plan by what it
owes, so it answers what share of the money is protected rather than what share of
the institutions. A policy that fixes many small plans and no large ones looks very
different under the two.

The curves are drawn on a fine axis by interpolating each plan's own risk curve
between grid points, so the steps come from plans crossing the target one by one.

In [ ]:
fine = np.linspace(0, GRID_DELTAS[-1], 200)
interp = np.vstack([np.interp(fine, GRID_DELTAS, risk.loc[p].to_numpy(dtype=float))
                    for p in GRID_PLANS])            # (n_plans, n_fine)

fig, axes = plt.subplots(1, 2, figsize=(13.5, 5.5), sharex=True)
colors = plt.cm.viridis(np.linspace(0.1, 0.85, len(RISK_TARGETS)))

for ax, weighted in zip(axes, (False, True)):
    for t_, c in zip(RISK_TARGETS, colors):
        under = interp <= t_
        if weighted:
            share = (under * LIAB_W[:, None]).sum(axis=0) / LIAB_W.sum()
        else:
            share = under.mean(axis=0)
        ax.plot(fine, share * 100, color=c, linewidth=2, label=f"<= {t_:.0%}")
    ax.set_xlabel("Permanent contribution increase (pp of payroll)")
    ax.set_ylim(0, 102)
    ax.legend(fontsize=8, title="P(exhaustion)", loc="lower right")
axes[0].set_ylabel("Plans reaching the level (%)")
axes[0].set_title("Unweighted")
axes[1].set_ylabel("Liability at inception in plans reaching the level (% of total)")
axes[1].set_title("Weighted by liability at inception")
plt.tight_layout()
plt.show()

rows = []
for d in GRID_DELTAS:
    r_ = {"added_pp": d}
    col = risk[d].to_numpy(dtype=float)
    for t_ in RISK_TARGETS:
        under = col <= t_
        r_[f"plans_under_{t_:g}"] = int(under.sum())
        r_[f"liab_share_under_{t_:g}"] = float((under * LIAB_W).sum() / LIAB_W.sum())
    rows.append(r_)
display(pd.DataFrame(rows).round(3))

---
### Per-Plan Detail

#### Single-Plan Forecast And Cash Flows

Select one plan: historical official funded ratios (black) followed by the simulated funding-ratio fan over the full usable projection, then mean projected cash inflows and outflows. Change `PLAN` to inspect a different plan.

In [ ]:
# Defaults to the largest plan by liability. It used to default to whichever
# plan sorted first alphabetically, which put AZ06 here for no reason at all.
PLAN = plan_metrics.nlargest(1, "liability_billion")["plan"].iloc[0]
# PLAN = "CA10"   # set explicitly to inspect a different plan

result = results[PLAN]
plan_fr = result.funding_ratio().iloc[:N_PROJ].to_numpy(dtype="float64")
plan_fan = fan_table(np.asarray(result.years(N_PROJ)), plan_fr)

fig, ax = plt.subplots(figsize=(11, 5.5))
actual = ra.actual_funding_ratio(ppd, result.ppid)
if not actual.empty:
    a = actual.loc[actual["fy"].between(2002, BASE_YEAR)]
    ax.plot(a["fy"], a["funding_ratio"], color="black", marker="o", markersize=3,
            label="Historical official")
plot_fan(ax, plan_fan, f"{PLAN} funding ratio")
ax.axvline(BASE_YEAR, color="0.4", linestyle=":", linewidth=1)
ax.axhline(1.0, color="0.4", linestyle="--", linewidth=1)
ax.set_title(f"Funding ratio: {PLAN} — history and simulated forecast")
ax.set_xlabel("Fiscal year")
ax.set_ylabel("Funding ratio")
ax.legend()
plt.show()

fig, ax = ra.plot_cashflow_dynamics(result, graph_years=N_PROJ)
plt.show()

display(plan_fan.head(10).round(3))

#### Largest Plans: Funding-Ratio Fans

Funding-ratio forecast fans for the five largest plans by GASB liability (mean line, 5-95 percentile band). These plans dominate the dollar aggregates in Part 2.

In [ ]:
top_plans = plan_metrics.nlargest(5, "liability_billion")["plan"].tolist()

fig, ax = plt.subplots(figsize=(11, 6))
for plan in top_plans:
    s = ra.forecast_summary(results[plan], graph_years=N_PROJ)
    line, = ax.plot(s["year"], s["mean"], label=plan)
    ax.fill_between(s["year"], s["q05"], s["q95"], alpha=0.10, color=line.get_color())
ax.axhline(1.0, color="0.4", linestyle="--", linewidth=1)
ax.set_title("Funding-ratio fans, five largest plans by liability")
ax.set_xlabel("Fiscal year")
ax.set_ylabel("Funding ratio")
ax.legend()
plt.show()

#### Model Liability Validation Against CAFR AAL

Compares the model's saved AAL estimate with the official CAFR/GASB AAL recorded in the result file. Points far from the 45-degree line indicate plans where the model liability differs materially from the reported liability. The histogram shows the percent-difference diagnostic across plans.

In [ ]:
validation = plan_metrics.dropna(subset=["model_aal_billion", "cafr_aal_billion"])[["plan", "PlanName", "model_aal_billion", "cafr_aal_billion", "percent_difference"]].copy()
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(validation["cafr_aal_billion"], validation["model_aal_billion"], alpha=0.7)
limit = max(validation["cafr_aal_billion"].max(), validation["model_aal_billion"].max())
ax.plot([0, limit], [0, limit], color="black", linestyle="--", linewidth=1)
ax.set_title("Model AAL vs CAFR AAL")
ax.set_xlabel("CAFR AAL, billions")
ax.set_ylabel("Model AAL, billions")
plt.show()

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(validation["percent_difference"].dropna(), bins=20, alpha=0.7)
ax.axvline(0, color="black", linestyle="--", linewidth=1)
ax.set_title("Percent-difference diagnostic: (Model AAL - CAFR AAL) / CAFR AAL")
ax.set_xlabel("Percent difference")
ax.set_ylabel("Plans")
plt.show()

display(validation.sort_values("percent_difference").round(3))

## Too complex / superfluous

Sections that are either re-expressions of something already shown above, or
were added as exploration and do not map to anything the V6 list asks for.

#### Funding-Ratio Distribution At Selected Horizons

Each row shows a plan's simulated funding-ratio distribution — thin line 5-95 percentile, thick line 20-80 percentile, tick at the median — at the `GRAPH_YEARS` mark (left) and at the end of the usable projection (right), sorted by median. The x-axis is capped at 3 for readability; upper tails of well-funded plans extend beyond it. The pooled histogram below shows all plan-path funding ratios at the shorter horizon. Full per-plan stats (including `prob_depleted` and threshold probabilities) remain available as `terminal_short` / `terminal_long`.

In [ ]:
terminal_short = ra.terminal_risk_table(results, year_offset=GRAPH_YEARS - 1, thresholds=(0.4, 0.6, 0.8, 1.0))
terminal_long = ra.terminal_risk_table(results, year_offset=N_PROJ - 1, thresholds=(0.4, 0.6, 0.8, 1.0))

# WHY IS THE CAP AT 3? MODIFIED TO 30
def plot_fr_intervals(table, title, ax, x_cap=None):
    # Cap derived from the panel's own data rather than hardcoded: the two
    # panels are decades apart and a shared cap flattens one of them. The 95th
    # percentile across plans, rounded up, keeps every median and 20-80 band
    # visible while cutting the runaway upper tail.
    if x_cap is None:
        # Clip to the MEDIAN 95th percentile across plans. Using a high
        # percentile of q95 lets two or three runaway plans set the axis and
        # flattens the rest, which is what it was doing.
        x_cap = float(np.ceil(np.nanmedian(table["q95"])))
    t = table.sort_values("median").reset_index(drop=True)
    y = np.arange(len(t))
    ax.hlines(y, t["q05"].clip(upper=x_cap), t["q95"].clip(upper=x_cap),
              color="tab:blue", alpha=0.35, linewidth=2, label="5-95 pct")
    ax.hlines(y, t["q20"].clip(upper=x_cap), t["q80"].clip(upper=x_cap),
              color="tab:blue", alpha=0.9, linewidth=4, label="20-80 pct")
    ax.plot(t["median"].clip(upper=x_cap), y, "k|", markersize=7, label="Median")
    ax.axvline(1.0, color="0.4", linestyle="--", linewidth=1)
    ax.set_yticks(y)
    ax.set_yticklabels(t["plan"], fontsize=8)
    ax.set_xlim(-0.05, x_cap + 0.05)
    ax.set_xlabel("Funding ratio")
    ax.set_title(title)


fig, axes = plt.subplots(1, 2, figsize=(13, max(6, len(terminal_short) * 0.30)))
fig.suptitle("Distribution of the funded ratio across simulated paths", fontsize=13)
plot_fr_intervals(terminal_short, f"Fiscal {int(terminal_short['year'].iloc[0])}", axes[0])
plot_fr_intervals(terminal_long, f"Fiscal {int(terminal_long['year'].iloc[0])}", axes[1])
axes[0].legend(loc="lower right")
plt.tight_layout()
plt.show()

# this plots the terminal distribution but for some reason uses 2036, which is NOT TERMINAL
_yr = int(terminal_long["year"].iloc[0])
fig, ax = ra.plot_terminal_distribution(results, year_offset=GRAPH_YEARS - 1)
ax.set_title(f"Funded ratio, all plans and paths, fiscal {_yr}")
ax.set_xlabel("Funded ratio (assets / accrued liability)")
plt.show()

#### Distress-Probability Heatmap

Each cell is the probability that a plan's funding ratio is below 0.4 in that projection year — deep distress, near the point where exhaustion becomes hard to avoid. Plans are sorted by overall distress probability. (For a central-tendency view, the module's `ra.plot_plan_heatmap(results, statistic="q50")` is still available.)

In [ ]:
DISTRESS_THRESHOLD = 0.4
risk_full = ra.threshold_risk_over_time(results, thresholds=(DISTRESS_THRESHOLD,), graph_years=N_PROJ)
heat = risk_full.pivot_table(index="plan", columns="year", values="probability")
heat = heat.loc[heat.mean(axis=1).sort_values(ascending=False).index]

fig, ax = plt.subplots(figsize=(12, max(6, len(heat) * 0.28)))
sns.heatmap(heat, ax=ax, cmap="Reds", vmin=0, vmax=1,
            cbar_kws={"label": f"P(funding ratio < {DISTRESS_THRESHOLD})"})
ax.set_title(f"Probability of funding ratio < {DISTRESS_THRESHOLD} by plan and year")
ax.set_xlabel("Fiscal year")
ax.set_ylabel("Plan")
plt.show()

#### Exhaustion Risk By Contribution Rate

Relationship between each plan's current contribution rate (total pension contributions divided by payroll) and its simulated 20-year exhaustion probability. Descriptive, not causal: high contribution rates may reflect plan stress rather than preventing it.

In [ ]:
contribution_scatter = exhaustion.merge(
    plan_metrics[["plan", "PlanName", "official_funded_ratio", "liability_billion", "unfunded_liabilities_payroll", "contribution_rate"]],
    on="plan",
    suffixes=("", "_metric"),
)
plot_df = contribution_scatter.dropna(subset=["contribution_rate", EXH_COLS[SCATTER_HORIZON], "liability_billion"])

fig, ax = plt.subplots(figsize=(10, 6))
sizes = 35 + 6 * np.sqrt(plot_df["liability_billion"].clip(lower=0))
ax.scatter(plot_df["contribution_rate"], plot_df[EXH_COLS[SCATTER_HORIZON]], s=sizes, alpha=0.65)
sns.regplot(data=plot_df, x="contribution_rate", y=EXH_COLS[SCATTER_HORIZON], scatter=False, ax=ax, color="black", line_kws={"linewidth": 1})
for _, row in plot_df.sort_values(EXH_COLS[SCATTER_HORIZON], ascending=False).head(8).iterrows():
    ax.annotate(row["plan"], (row["contribution_rate"], row[EXH_COLS[SCATTER_HORIZON]]), fontsize=9, xytext=(4, 4), textcoords="offset points")
ax.set_title(f"P(exhaustion by {BASE_YEAR + SCATTER_HORIZON}) vs contribution rate")
ax.set_xlabel("Total pension contributions / payroll")
ax.set_ylabel(f"P(exhaustion by {BASE_YEAR + SCATTER_HORIZON})")
ax.set_ylim(-0.02, min(1.02, max(0.1, plot_df[EXH_COLS[SCATTER_HORIZON]].max() + 0.08)))
plt.show()

display(
    plot_df[["plan", "PlanName", "contribution_rate", EXH_COLS[10], EXH_COLS[SCATTER_HORIZON], EXH_COLS[30], "liability_billion", "official_funded_ratio", "unfunded_liabilities_payroll"]]
    .sort_values(EXH_COLS[SCATTER_HORIZON], ascending=False)
    .round(3)
)

#### Conditional Severity: How Big Is The Hole, Given That It Opens

Exhaustion probability says how often a plan runs out of assets. It says nothing
about how much money is involved when it does. This section measures the size.

Two quantities, both conditional on a path in which the plan does run out:

- **Liability outstanding at exhaustion** — the accrued liability still on the
  books in the year assets first reach zero. Assets are floored at zero in the
  simulation, so in that year the entire remaining liability is unfunded.
- **Benefits not covered by contributions** — once assets are gone, benefits still have to be
  paid out of current money. For every year in which a plan holds zero assets,
  this is the amount by which benefit payments exceed contributions coming in.
  It is reported both undiscounted and discounted back to the base year at the
  plan's own discount rate.

Note what varies and what does not. In these runs the liability and cash-flow
paths are deterministic — one column, repeated across simulations — so all the
variation across paths comes from asset returns. The severity measures are
therefore driven by *when* a plan exhausts, not by a separate random liability.

The final figure is the one worth dwelling on: probability against conditional
severity. A plan that fails rarely but expensively and a plan that fails often
but cheaply are very different objects, and a single exhaustion probability
cannot tell them apart.

In [ ]:
# Liability and cash-flow matrices are recycled from a single deterministic
# column, so column 0 represents every path. Verified rather than assumed.
def _deterministic_column(result, name):
    arr = result.matrix(name).to_numpy(dtype="float64")
    if not np.allclose(arr, arr[:, [0]], equal_nan=True):
        raise ValueError(f"{result.plan}: {name} varies across simulations; "
                         "the severity section assumes it does not.")
    return arr[:, 0]


def severity_paths(result):
    """Per-path pay-go shortfall, plus the liability outstanding at exhaustion."""
    assets = result.matrix("Assets").to_numpy(dtype="float64")
    aal = _deterministic_column(result, "AAL")
    outflow = _deterministic_column(result, "cash_outflows")
    inflow = _deterministic_column(result, "cash_inflows")
    rate = float(result.scalars.get("discountrate", np.nan))

    # Liability-side years only: AAL and the flows stop one year before assets.
    t = np.arange(N_PROJ)
    depleted = assets[:N_PROJ, :] <= 0
    annual_gap = np.maximum(outflow[:N_PROJ] - inflow[:N_PROJ], 0.0)[:, None] * depleted
    discount = (1.0 + rate) ** (-t.astype(float))
    return {
        "annual_gap": annual_gap,                              # (N_PROJ, n_sim)
        "total_gap": annual_gap.sum(axis=0),
        "total_gap_pv": (annual_gap * discount[:, None]).sum(axis=0),
        "aal": aal,
        "depleted": depleted,
    }


sev = {plan: severity_paths(result) for plan, result in sorted(results.items())}

rows = []
for plan, result in sorted(results.items()):
    s = sev[plan]
    offsets = exhaustion_year(result)
    # Liability outstanding at exhaustion needs an AAL value, so it is defined
    # only for exhaustions inside the liability horizon.
    # Liability outstanding at exhaustion needs an AAL value at that row. When
    # the two horizons agree this excludes nothing; on pre-2026-08-04 runs it
    # drops the exhaustions that fall in the extra asset year.
    within = np.isfinite(offsets) & (offsets <= N_PROJ - 1)
    idx = offsets[within].astype(int)
    liab_at_exh = s["aal"][idx] / 1e9 if idx.size else np.array([])
    exhausts = np.isfinite(offsets)
    gap_pv = s["total_gap_pv"][exhausts] / 1e9
    liability = plan_metrics.set_index("plan").loc[plan, "liability_billion"]
    rows.append({
        "plan": plan,
        "liability_billion": liability,
        "prob_exhaust": float(exhausts.mean()),
        "median_liability_at_exhaustion_billion": float(np.median(liab_at_exh)) if liab_at_exh.size else np.nan,
        "median_paygo_pv_billion": float(np.median(gap_pv)) if gap_pv.size else np.nan,
        "p90_paygo_pv_billion": float(np.quantile(gap_pv, 0.90)) if gap_pv.size else np.nan,
        "median_paygo_pv_pct_liability": float(np.median(gap_pv) / liability * 100) if gap_pv.size and liability > 0 else np.nan,
        "median_years_depleted_if_exhausts": float(np.median(s["depleted"].sum(axis=0)[exhausts])) if exhausts.any() else np.nan,
    })
severity = pd.DataFrame(rows)

# --- aggregate pay-go requirement across all plans, path by path -------------
agg_gap = np.zeros_like(sev[next(iter(sev))]["annual_gap"])
for s in sev.values():
    agg_gap = agg_gap + s["annual_gap"]
gap_years = np.arange(BASE_YEAR, BASE_YEAR + N_PROJ)
gap_fan = fan_table(gap_years, agg_gap / 1e9)

fig, ax = plt.subplots(figsize=(10, 5))
plot_fan(ax, gap_fan, "Benefits not covered by contributions", color="tab:purple")
ax.set_title("Benefits not covered by contributions in plans holding zero assets")
ax.set_xlabel("Fiscal year")
ax.set_ylabel("Billions of dollars per year")
ax.legend()
plt.show()

# --- probability against conditional severity -------------------------------
p = severity.dropna(subset=["median_paygo_pv_billion"])
p = p.loc[p["prob_exhaust"] > 0]
fig, ax = plt.subplots(figsize=(10, 6))
sizes = 35 + 6 * np.sqrt(p["liability_billion"].clip(lower=0))
ax.scatter(p["prob_exhaust"], p["median_paygo_pv_billion"], s=sizes, alpha=0.65, color="tab:purple")
for _, row in p.nlargest(8, "median_paygo_pv_billion").iterrows():
    ax.annotate(row["plan"], (row["prob_exhaust"], row["median_paygo_pv_billion"]),
                fontsize=9, xytext=(4, 4), textcoords="offset points")
ax.set_xlabel("P(exhaustion by %d)" % ASSET_LAST_YEAR)
ax.set_ylabel("Median pay-go shortfall if it happens (PV, $bn)")
ax.set_title("Probability against conditional severity (bubble = liability)")
plt.show()

display(severity.sort_values("median_paygo_pv_billion", ascending=False).round(3))

#### Average Funding-Ratio Forecast

The black line is the historical equal-weighted average of official funded ratios across the plans in the run. The fan is the simulated continuation: the **equal-weighted average funding ratio across plans, computed within each Monte Carlo path** and then summarized across paths. Per-path cross-plan averaging is valid because all plans share common market shocks. This equal-plan view describes the typical plan and complements the dollar-aggregate fan above, which is dominated by the largest plans.

In [ ]:
# Equal-weighted average funding ratio per path (cross-plan average, valid under common shocks)
n_sims_common = min(r.n_simulations for r in results.values())
fr_stack = np.stack([
    r.funding_ratio().iloc[:N_PROJ, :n_sims_common].to_numpy(dtype="float64")
    for r in results.values()
], axis=0)
avg_fr_paths = np.nanmean(fr_stack, axis=0)
fr_years = np.asarray(next(iter(results.values())).years(N_PROJ))
avg_fr_fan = fan_table(fr_years, avg_fr_paths)

hist = historical_official_funding(ppd, plan_metrics["ppid"].dropna().astype(int).tolist())
hist = hist.loc[hist["fy"].between(2002, BASE_YEAR)]

fig, ax = plt.subplots(figsize=(11, 5.5))
ax.plot(hist["fy"], hist["equal_weighted"], color="black", marker="o", markersize=3,
        label="Historical official (equal-weighted)")
plot_fan(ax, avg_fr_fan, "Average funding ratio")
ax.axvline(BASE_YEAR, color="0.4", linestyle=":", linewidth=1)
ax.axhline(1.0, color="0.4", linestyle="--", linewidth=1)
ax.set_title("Equal-weighted average funding ratio: history and simulated forecast")
ax.set_xlabel("Fiscal year")
ax.set_ylabel("Funding ratio")
ax.legend()
plt.show()

display(avg_fr_fan.head(10).round(3))

##### The same aggregates as bounded quantities

The three fans above all plot a **level that compounds**. Assets grow at roughly a
constant expected rate, so the central line traces an exponential and the bands are
the same exponential with a different base. The whole chart then encodes essentially
one number — dispersion in the growth rate — and the eye reads "things get large",
which is not a finding. The upper tail also sets the axis scale, compressing
everything else into a strip near zero.

Below is the same information as **probabilities**, which are bounded between 0 and
1 and so have somewhere for dispersion to go: at every point on the horizontal axis
the vertical position means something on its own, rather than only by comparison
with the line beside it.

The originals are kept directly above for comparison rather than replaced.

In [ ]:
# --- items 20-23: bounded views of the aggregates ---------------------------
agg_ratio = np.divide(agg_assets, agg_aal,
                      out=np.full_like(agg_assets, np.nan), where=agg_aal > 0)
agg_short_bn = (agg_aal - agg_assets) / 1e9          # positive = shortfall

fig, axes = plt.subplots(1, 2, figsize=(13.5, 5))

# 20. aggregate funded ratio -> probability of being below a level
ax = axes[0]
for lvl, c in zip((1.0, 0.8, 0.6), plt.cm.viridis(np.linspace(0.15, 0.8, 3))):
    ax.plot(agg_years, np.nanmean(agg_ratio < lvl, axis=1) * 100,
            color=c, linewidth=2, label=f"below {lvl:g}")
ax.set_xlabel("Fiscal year")
ax.set_ylabel("Probability (%)")
ax.set_title("P(aggregate funded ratio below threshold)")
ax.set_ylim(-2, 102)
ax.legend(fontsize=8, title="Funded ratio", loc="upper left")

# 21. aggregate shortfall -> probability of exceeding a size
ax = axes[1]
for thr, c in zip((0, 250, 500, 1000), plt.cm.plasma(np.linspace(0.1, 0.75, 4))):
    ax.plot(agg_years, np.nanmean(agg_short_bn > thr, axis=1) * 100,
            color=c, linewidth=2, label=("any shortfall" if thr == 0 else f"over ${thr:g}bn"))
ax.set_xlabel("Fiscal year")
ax.set_ylabel("Probability (%)")
ax.set_title("P(aggregate shortfall above threshold)")
ax.set_ylim(-2, 102)
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

# 22. the equal-weighted average, median only, against history
fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(hist["fy"], hist["equal_weighted"], color="black", marker="o", markersize=3,
        label="Reported, equal-weighted across plans")
ax.plot(fr_years, np.nanmedian(avg_fr_paths, axis=1), color="tab:blue", linewidth=2,
        label="Simulated median")
ax.plot(fr_years, np.nanpercentile(avg_fr_paths, 25, axis=1), color="tab:blue",
        linewidth=1, linestyle="--", label="25th / 75th percentile")
ax.plot(fr_years, np.nanpercentile(avg_fr_paths, 75, axis=1), color="tab:blue",
        linewidth=1, linestyle="--")
ax.axvline(BASE_YEAR, color="0.4", linestyle=":", linewidth=1)
ax.axhline(1.0, color="0.4", linestyle="--", linewidth=1)
ax.set_xlabel("Fiscal year")
ax.set_ylabel("Funded ratio, mean across plans")
ax.set_title("Funded ratio, mean across plans")
ax.legend(fontsize=8)
plt.show()

# 25. log-scale view of the level fan, so band width reads as growth-rate spread
fig, ax = plt.subplots(figsize=(10, 5))
for q, st in ((5, ":"), (25, "--"), (50, "-"), (75, "--"), (95, ":")):
    ax.plot(agg_years, np.nanpercentile(agg_ratio, q, axis=1),
            color="tab:blue", linestyle=st, linewidth=1.6 if q == 50 else 1.1,
            label=f"{q}th percentile")
ax.set_yscale("log")
ax.axhline(1.0, color="0.4", linestyle="--", linewidth=1)
ax.set_xlabel("Fiscal year")
ax.set_ylabel("Assets / liability, log scale")
ax.set_title("Aggregate funded ratio, log scale")
ax.legend(fontsize=8)
plt.show()

##### The GDP views as probabilities

Same treatment for the GDP-normalised aggregate: a probability of the shortfall
exceeding a given share of GDP, which is bounded and reads directly, rather than a
fan of a compounding level. The assets-and-liability chart above is mean-only, so a
quartile band is added; that one is worth keeping as a level, because liability
falling as a share of GDP is a statement the upper tail does not drive.

In [ ]:
if gdp_series is not None:
    short_gdp = 100 * (agg_aal - agg_assets) / 1e9 / gdp_path[:, None]

    fig, axes = plt.subplots(1, 2, figsize=(13.5, 5))

    # 23. probability the shortfall exceeds a share of GDP
    ax = axes[0]
    for thr, c in zip((0, 2, 5, 10), plt.cm.plasma(np.linspace(0.1, 0.75, 4))):
        ax.plot(agg_years, np.nanmean(short_gdp > thr, axis=1) * 100,
                color=c, linewidth=2,
                label=("any shortfall" if thr == 0 else f"over {thr:g}% of GDP"))
    ax.set_xlabel("Fiscal year")
    ax.set_ylabel("Probability (%)")
    ax.set_title("P(aggregate shortfall above threshold, share of GDP)")
    ax.set_ylim(-2, 102)
    ax.legend(fontsize=8)

    # 24. assets and liability as a share of GDP, with a band
    ax = axes[1]
    for arr, name, c in ((agg_assets, "Assets", "tab:blue"),
                         (agg_aal, "Accrued liability", "tab:orange")):
        pct = 100 * arr / 1e9 / gdp_path[:, None]
        ax.plot(agg_years, np.nanmedian(pct, axis=1), color=c, linewidth=2, label=f"{name}, median")
        ax.fill_between(agg_years, np.nanpercentile(pct, 25, axis=1),
                        np.nanpercentile(pct, 75, axis=1), color=c, alpha=0.15)
    ax.set_xlabel("Fiscal year")
    ax.set_ylabel("Percent of projected GDP")
    ax.set_title("Assets and accrued liability, share of GDP")
    ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()
else:
    print("GDP views skipped: no FRED data in this session.")

#### How responsive each plan is

The same increase does very different things to different plans, and that is a
property worth isolating rather than leaving inside the curves.

The left panel is the total risk removed by the largest increase on the grid. The
right panel is the **marginal** effect: how much risk each additional percentage
point removes, at each point along the range. A flat line means the lever works
about equally hard everywhere; a rising line means contributions compound into one
another and later percentage points do more than earlier ones; a falling line means
the plan is running out of room because its risk is already near zero.

In [ ]:
total_drop = (risk[0.0] - risk[GRID_DELTAS[-1]]).sort_values()

fig, axes = plt.subplots(1, 2, figsize=(13.5, max(5.5, len(GRID_PLANS) * 0.22)))

ax = axes[0]
ax.barh(np.arange(len(total_drop)), total_drop * 100, color="tab:purple", alpha=0.8)
ax.set_yticks(np.arange(len(total_drop)))
ax.set_yticklabels(total_drop.index, fontsize=8)
ax.set_xlabel(f"Reduction in P(exhaustion), pp")
ax.set_title(f"Reduction in P(exhaustion) at +{GRID_DELTAS[-1]:g}pp")

ax = axes[1]
mid = (GRID_DELTAS[:-1] + GRID_DELTAS[1:]) / 2
step = np.diff(GRID_DELTAS)
# All 40 drawn identically, as in the curves above.
for p in GRID_PLANS:
    y = -np.diff(risk.loc[p].to_numpy(dtype=float)) / step
    ax.plot(mid, y * 100, color="0.7", linewidth=0.9, zorder=1)
wy = -np.diff((risk.T.to_numpy() @ LIAB_W) / LIAB_W.sum()) / step
ax.plot(mid, wy * 100, color="black", linewidth=3,
        label="All plans, weighted by liability at inception", zorder=4)
ax.set_xlabel("Contribution increase (pp of payroll)")
ax.set_ylabel("Reduction in P(exhaustion) per extra pp")
ax.set_title("Marginal reduction in P(exhaustion) per pp")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

resp = pd.DataFrame({
    "risk_at_0pp": risk[0.0],
    f"risk_at_{GRID_DELTAS[-1]:g}pp": risk[GRID_DELTAS[-1]],
    "total_risk_removed": risk[0.0] - risk[GRID_DELTAS[-1]],
    "removed_per_pp_avg": (risk[0.0] - risk[GRID_DELTAS[-1]]) / GRID_DELTAS[-1],
    "liability_billion": _liab,
})
resp["share_of_starting_risk_removed"] = np.where(
    resp.risk_at_0pp > 0, resp.total_risk_removed / resp.risk_at_0pp, np.nan)
display(resp.sort_values("total_risk_removed", ascending=False).round(4))

#### Joint failure under the policy

Everything above is per plan. This asks what the policy does to the thing that only
appears when plans are looked at together: **how many of them run out of assets in
the same market history.**

Because every scenario shares one set of market histories, a simulation column is
the same sequence of good and bad years at every contribution level. So the shift in
these distributions is the policy and nothing else.

The left panel is the whole distribution of how many plans have failed by the end,
at each contribution level. The right panel tracks its centre and its upper tail as
contributions rise — worth watching separately, because a policy can move the middle
without moving the tail, and the tail is where a simultaneous failure lives.

In [ ]:
counts = {d: ind[:, -1, :].sum(axis=0) for d, ind in grid_exhausted.items()}
liab_share = {d: (ind[:, -1, :] * LIAB_W[:, None]).sum(axis=0) / LIAB_W.sum()
              for d, ind in grid_exhausted.items()}

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

ax = axes[0]
colors = plt.cm.viridis(np.linspace(0.1, 0.9, len(GRID_DELTAS)))
k = np.arange(len(GRID_PLANS) + 1)
for (d, c_), col in zip(counts.items(), colors):
    dist = np.bincount(c_, minlength=len(GRID_PLANS) + 1) / c_.size
    ax.plot(k, dist * 100, color=col, linewidth=1.8, label=f"+{d:g}pp")
ax.set_xlabel("Plans exhausted")
ax.set_ylabel("Probability (%)")
ax.set_title(f"Distribution of plans exhausted by {ASSET_LAST_YEAR}")
ax.legend(fontsize=8)

ax = axes[1]
ds = np.array(sorted(counts))
mean_n = np.array([counts[d].mean() for d in ds])
p90_n = np.array([np.quantile(counts[d], 0.90) for d in ds])
p99_n = np.array([np.quantile(counts[d], 0.99) for d in ds])
ax.plot(ds, mean_n, marker="o", linewidth=2, label="mean")
ax.plot(ds, p90_n, marker="s", linewidth=2, label="90th percentile")
ax.plot(ds, p99_n, marker="^", linewidth=2, label="99th percentile")
ax.set_xlabel("Permanent contribution increase (pp of payroll)")
ax.set_ylabel("Plans exhausted")
ax.set_title("Plans exhausted: mean and upper percentiles")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(10, 5))
ls_mean = np.array([liab_share[d].mean() for d in ds]) * 100
ls_p90 = np.array([np.quantile(liab_share[d], 0.90) for d in ds]) * 100
ls_p99 = np.array([np.quantile(liab_share[d], 0.99) for d in ds]) * 100
ax.fill_between(ds, ls_mean, ls_p99, alpha=0.15, color="tab:red", label="mean to 99th pct")
ax.plot(ds, ls_mean, marker="o", color="tab:red", linewidth=2, label="mean")
ax.plot(ds, ls_p90, marker="s", color="tab:red", linewidth=1.5, linestyle="--", label="90th pct")
ax.plot(ds, ls_p99, marker="^", color="tab:red", linewidth=1.5, linestyle=":", label="99th pct")
ax.set_xlabel("Permanent contribution increase (pp of payroll)")
ax.set_ylabel("Liability at inception in those plans (% of total)")
ax.set_title("Liability in exhausted plans")
ax.legend(fontsize=8)
plt.show()

display(pd.DataFrame({
    "added_pp": ds,
    "mean_plans_failed": mean_n,
    "p90_plans_failed": p90_n,
    "p99_plans_failed": p99_n,
    "mean_liab_share_failed": [liab_share[d].mean() for d in ds],
    "p99_liab_share_failed": [np.quantile(liab_share[d], 0.99) for d in ds],
}).round(3))

## Deleted

Slated for removal. Kept here only so the decision is visible and reversible.

---
### Cohort Structure (Per-Tier Decomposition)

The model divides each plan's active members into **tiers** — groups hired under
different benefit rules, created whenever a plan changed what it promised new
employees. Members hired before a reform stay on the old terms; members hired
after are on the new ones. Retirees already collecting benefits are carried as a
separate group again.

This section splits each plan's liability and benefit payments into those
groups. It answers questions the plan-level figures cannot: how much of what a
plan owes is owed to people already retired, how much to long-serving staff
under older and more generous terms, and how much to recent hires.

**Three things to be clear about before reading it.**

**Where the numbers come from.** The per-tier paths are saved in each plan's
deterministic result file but are *not* included in the parquet bundle the rest
of this notebook reads, so this section opens the result files directly. No
re-run is needed; the data has been there all along.

**These are hire-date groups, not birth cohorts.** A tier is defined by when
someone was hired and therefore which benefit rules apply to them, not by when
they were born. The two are related but they are not the same thing, and a tier
contains people of many ages. The latest tier boundary anywhere in the input
data is 1 July 2018, so nothing enacted since is represented.

**This decomposition is deterministic.** Each tier carries a single projected
path, not a distribution. Assets are simulated at the level of the whole plan,
so there is no simulated outcome belonging to one tier rather than another.
Turning this into a distribution *across* cohorts — which is what the project's
framing ultimately wants — requires deciding how a plan-level asset shortfall
should be attributed to the groups within it. That is a modelling decision, not
a plotting one, and it has not been taken.

In [ ]:
import pickle as _pickle


def load_tier_paths(root, run_tag, plans):
    """Per-tier liability and benefit-outflow paths from the deterministic files.

    MainRes is keyed 1..6 by tier, each holding [AAL, outflow, inflow, PVFB, NC];
    RetRes holds [AAL, outflow] for members already retired at the base year.
    """
    out = {}
    for plan in plans:
        path = (Path(root) / "Results" / "Runs" / run_tag / plan
                / f"{plan}_detAL_{run_tag}.pkl")
        with path.open("rb") as handle:
            payload = _pickle.load(handle)
        groups = {}
        for tier, arrays in sorted(payload["MainRes"].items()):
            aal = np.asarray(arrays[0], dtype="float64").reshape(-1)
            outflow = np.asarray(arrays[1], dtype="float64").reshape(-1)
            if aal[0] > 0:                      # tiers with no members are all zero
                groups[f"Tier {tier}"] = {"aal": aal, "outflow": outflow}
        ret = payload["RetRes"]
        groups["Already retired"] = {
            "aal": np.asarray(ret[0], dtype="float64").reshape(-1),
            "outflow": np.asarray(ret[1], dtype="float64").reshape(-1),
        }
        # The parts must reconstruct the whole, or the decomposition is wrong.
        total = sum(g["aal"] for g in groups.values())
        saved = np.asarray(payload["AAL"], dtype="float64")[:, 0]
        if not np.allclose(total, saved, rtol=1e-9, atol=1.0):
            raise ValueError(f"{plan}: tier AALs do not sum to the saved total AAL.")
        out[plan] = groups
    return out


tier_paths = load_tier_paths(ROOT, RUN_TAG, sorted(results))
print(f"Loaded per-tier paths for {len(tier_paths)} plans; "
      f"tier counts: {sorted({len(v) for v in tier_paths.values()})} groups per plan "
      "(including the already-retired group)")

tier_rows = []
for plan, groups in tier_paths.items():
    total = sum(g["aal"][0] for g in groups.values())
    for name, g in groups.items():
        tier_rows.append({"plan": plan, "group": name,
                          "aal_billion": g["aal"][0] / 1e9,
                          "share": g["aal"][0] / total if total > 0 else np.nan})
tier_frame = pd.DataFrame(tier_rows)

order = plan_metrics.sort_values("liability_billion")["plan"].tolist()
pivot = (tier_frame.pivot_table(index="plan", columns="group", values="share")
         .reindex(order).fillna(0.0))
group_order = ["Already retired"] + [c for c in sorted(pivot.columns) if c != "Already retired"]
pivot = pivot[group_order]

fig, ax = plt.subplots(figsize=(12.5, max(6, len(pivot) * 0.30)))
left = np.zeros(len(pivot))
colors = plt.cm.viridis(np.linspace(0.15, 0.9, len(group_order)))
for colour, name in zip(colors, group_order):
    ax.barh(pivot.index, pivot[name] * 100, left=left * 100, label=name, color=colour)
    left = left + pivot[name].to_numpy()
ax.set_xlabel("Share of the plan's base-year accrued liability (%)")
ax.set_title("Who each plan owes: liability split by benefit-rule group")
ax.set_xlim(0, 100)
ax.legend(loc="center left", bbox_to_anchor=(1.01, 0.5), fontsize=8,
          title="Benefit-rule group", frameon=False)
plt.tight_layout()
plt.show()

aggregate_share = (tier_frame.groupby("group")["aal_billion"].sum()
                   .sort_values(ascending=False).to_frame("aal_billion"))
aggregate_share["share_of_all_plans"] = (aggregate_share["aal_billion"]
                                         / aggregate_share["aal_billion"].sum())
display(aggregate_share.round(3))

# --- benefit payments by group over time, largest plan ----------------------
big_plan = plan_metrics.nlargest(1, "liability_billion")["plan"].iloc[0]
groups = tier_paths[big_plan]
years_c = np.arange(BASE_YEAR, BASE_YEAR + N_PROJ)
fig, ax = plt.subplots(figsize=(11, 5.5))
stack = [groups[name]["outflow"][:N_PROJ] / 1e9 for name in group_order if name in groups]
labels = [name for name in group_order if name in groups]
ax.stackplot(years_c, *stack, labels=labels,
             colors=plt.cm.viridis(np.linspace(0.15, 0.9, len(labels))))
ax.set_title(f"{big_plan}: projected benefit payments by benefit-rule group")
ax.set_xlabel("Fiscal year")
ax.set_ylabel("Billions of dollars per year")
ax.legend(loc="upper left", fontsize=8)
plt.show()

display(tier_frame.loc[tier_frame["plan"] == big_plan].round(3))

---
### Items Requiring New Scenario Runs

The current result files support everything above. The following require new simulation scenarios, not more post-processing:

- **Contribution-policy counterfactuals:** the permanent contribution increase needed to hit target exhaustion probabilities (for example 0.5%, 1%, 3%) requires rerunning the asset stage with an explicit contribution-policy parameter. At `num_sim = 10000` the probability grid (0.01 percentage points) is fine enough; the missing piece is the policy lever in the simulation, not precision.
- **Waiting-period stabilization scenarios:** permanent contribution increases starting immediately versus after 5, 10, or 15 years — same policy engine as above, plus a start-delay parameter.
- **Investment-strategy counterfactuals:** reruns under alternative asset allocations or return assumptions (for example de-risking paths). The shared market-shock matrix makes these directly comparable path-by-path: the same market history can be replayed under each policy.
- **Model-consistent AAA liability revaluation:** recomputing AAL with an AAA discount rate inside the liability model (not just discounting projected cash flows, as in the exploratory section above).
- **Tier-level reform comparison (planned rework):** the earlier descriptive reform section was removed; the replacement should compare each plan's pre-change versus post-change tier provisions directly (matched to the model's tier structure), rather than the earliest-vs-latest workbook diff used before.
- **No-reform counterfactuals:** rerunning the liability model with pre-reform benefit rules for plans that changed tiers after 2007, to quantify what reforms saved.

Check the run manifest at the top before using tables as final results; rerun any missing plan outputs first.